# NFL Spread Prediction Data Exploration Using nflverse

## UMBC Data Science Capstone Proposal Notebook

This notebook pulls NFL game, betting, and play-by-play data from the nflverse ecosystem using the Python package `nfl_data_py`.

The project goal is to build a game-level modeling dataset where each row represents one NFL game. The primary target variable is the home team's final point margin. The model will eventually predict the expected home margin, compare that prediction to the sportsbook closing spread, and use a threshold-based rule to decide whether a simulated bet should be placed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

print("Packages imported successfully.")

Packages imported successfully.


In [ ]:
# Project folder structure inside Colab
project_root = Path("/content/nfl_spread_capstone")

data_dir = project_root / "data"
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"
notebooks_dir = project_root / "notebooks"
docs_dir = project_root / "docs"

# Create folders
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
notebooks_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

print("Project folders created:")
print(project_root)
print(raw_dir)
print(processed_dir)
print(notebooks_dir)
print(docs_dir)

Project folders created:
/content/nfl_spread_capstone
/content/nfl_spread_capstone/data/raw
/content/nfl_spread_capstone/data/processed
/content/nfl_spread_capstone/notebooks
/content/nfl_spread_capstone/docs


In [ ]:
schedule_url = "https://github.com/nflverse/nfldata/raw/master/data/games.csv"

schedules = pd.read_csv(schedule_url)

schedules.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,1999_01_MIN_ATL,1999,REG,1,1999-09-12,Sunday,NaN,MIN,17.000,ATL,14.000,Home,-3.000,31.000,0.000,1999091210,598.000,NaN,199909120atl,NaN,190912001.000,NaN,7,7,NaN,NaN,-4.000,NaN,NaN,49.000,NaN,NaN,0,dome,astroturf,NaN,NaN,00-0003761,00-0002876,Randall Cunningham,Chris Chandler,Dennis Green,Dan Reeves,Gerry Austin,ATL00,Georgia Dome
1,1999_01_KC_CHI,1999,REG,1,1999-09-12,Sunday,NaN,KC,17.000,CHI,20.000,Home,3.000,37.000,0.000,1999091206,597.000,NaN,199909120chi,NaN,190912003.000,NaN,7,7,NaN,NaN,-3.000,NaN,NaN,38.000,NaN,NaN,0,outdoors,grass,80.000,12.000,00-0006300,00-0010560,Elvis Grbac,Shane Matthews,Gunther Cunningham,Dick Jauron,Phil Luckett,CHI98,Soldier Field
2,1999_01_PIT_CLE,1999,REG,1,1999-09-12,Sunday,NaN,PIT,43.000,CLE,0.000,Home,-43.000,43.000,0.000,1999091213,604.000,NaN,199909120cle,NaN,190912005.000,NaN,7,7,NaN,NaN,-6.000,NaN,NaN,37.000,NaN,NaN,1,outdoors,grass,78.000,12.000,00-0015700,00-0004230,Kordell Stewart,Ty Detmer,Bill Cowher,Chris Palmer,Bob McElwee,CLE00,Cleveland Browns Stadium
3,1999_01_OAK_GB,1999,REG,1,1999-09-12,Sunday,NaN,OAK,24.000,GB,28.000,Home,4.000,52.000,0.000,1999091208,602.000,NaN,199909120gnb,NaN,190912009.000,NaN,7,7,NaN,NaN,9.000,NaN,NaN,43.000,NaN,NaN,0,outdoors,grass,67.000,10.000,00-0005741,00-0005106,Rich Gannon,Brett Favre,Jon Gruden,Ray Rhodes,Tony Corrente,GNB00,Lambeau Field
4,1999_01_BUF_IND,1999,REG,1,1999-09-12,Sunday,NaN,BUF,14.000,IND,31.000,Home,17.000,45.000,0.000,1999091202,591.000,NaN,199909120clt,NaN,190912011.000,NaN,7,7,NaN,NaN,-3.000,NaN,NaN,45.500,NaN,NaN,1,dome,astroturf,NaN,NaN,00-0005363,00-0010346,Doug Flutie,Peyton Manning,Wade Phillips,Jim Mora,Ron Blum,IND99,RCA Dome


In [ ]:
schedules["season"].min(), schedules["season"].max()

(1999, 2026)

In [ ]:
schedules.columns.tolist()

['game_id',
 'season',
 'game_type',
 'week',
 'gameday',
 'weekday',
 'gametime',
 'away_team',
 'away_score',
 'home_team',
 'home_score',
 'location',
 'result',
 'total',
 'overtime',
 'old_game_id',
 'gsis',
 'nfl_detail_id',
 'pfr',
 'pff',
 'espn',
 'ftn',
 'away_rest',
 'home_rest',
 'away_moneyline',
 'home_moneyline',
 'spread_line',
 'away_spread_odds',
 'home_spread_odds',
 'total_line',
 'under_odds',
 'over_odds',
 'div_game',
 'roof',
 'surface',
 'temp',
 'wind',
 'away_qb_id',
 'home_qb_id',
 'away_qb_name',
 'home_qb_name',
 'away_coach',
 'home_coach',
 'referee',
 'stadium_id',
 'stadium']

In [ ]:
schedules_2010_2025 = schedules[
    (schedules["season"] >= 2010) &
    (schedules["season"] <= 2025) &
    (schedules["game_type"] == "REG")
].copy()

schedules_2010_2025.shape

(4175, 46)

In [ ]:
schedules_2010_2025.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
2913,2010_01_MIN_NO,2010,REG,1,2010-09-09,Thursday,20:30,MIN,9.000,NO,14.000,Home,5.000,23.000,0.000,2010090900,54863.000,NaN,201009090nor,1727.000,300909018.000,NaN,7,7,197.000,-220.000,4.500,-105.000,-103.000,48.500,-104.000,-106.000,0,dome,sportturf,NaN,NaN,00-0005106,00-0020531,Brett Favre,Drew Brees,Brad Childress,Sean Payton,Terry McAulay,NOR00,Louisiana Superdome
2914,2010_01_MIA_BUF,2010,REG,1,2010-09-12,Sunday,13:00,MIA,15.000,BUF,10.000,Home,-5.000,25.000,0.000,2010091201,54864.000,NaN,201009120buf,1729.000,300912002.000,NaN,7,7,-155.000,140.000,-3.000,-106.000,-102.000,39.500,-110.000,100.000,1,outdoors,astroplay,62.000,7.000,00-0026197,00-0025479,Chad Henne,Trent Edwards,Tony Sparano,Chan Gailey,Clete Blakeman,BUF00,Ralph Wilson Stadium
2915,2010_01_DET_CHI,2010,REG,1,2010-09-12,Sunday,13:00,DET,14.000,CHI,19.000,Home,5.000,33.000,0.000,2010091207,54865.000,NaN,201009120chi,1736.000,300912003.000,NaN,7,7,248.000,-280.000,6.500,103.000,-111.000,44.500,-105.000,-105.000,1,outdoors,grass,75.000,10.000,00-0026498,00-0024226,Matthew Stafford,Jay Cutler,Jim Schwartz,Lovie Smith,Gene Steratore,CHI98,Soldier Field
2916,2010_01_IND_HOU,2010,REG,1,2010-09-12,Sunday,13:00,IND,24.000,HOU,34.000,Home,10.000,58.000,0.000,2010091203,54866.000,NaN,201009120htx,1731.000,300912034.000,NaN,7,7,-117.000,106.000,-1.000,-110.000,102.000,47.500,-102.000,-108.000,1,closed,grass,NaN,NaN,00-0010346,00-0022787,Peyton Manning,Matt Schaub,Jim Caldwell,Gary Kubiak,Ed Hochuli,HOU00,Reliant Stadium
2917,2010_01_DEN_JAX,2010,REG,1,2010-09-12,Sunday,13:00,DEN,17.000,JAX,24.000,Home,7.000,41.000,0.000,2010091204,54867.000,NaN,201009120jax,1732.000,300912030.000,NaN,7,7,166.000,-185.000,3.000,109.000,-118.000,41.500,-110.000,100.000,0,outdoors,grass,90.000,10.000,00-0023541,00-0021231,Kyle Orton,David Garrard,Josh McDaniels,Jack Del Rio,Walt Coleman,JAX00,EverBank Field


In [ ]:
schedule_file = raw_dir / "nflverse_schedules_2010_2025.csv"

schedules_2010_2025.to_csv(schedule_file, index=False)

print(f"Saved schedule data to: {schedule_file}")

Saved schedule data to: /content/nfl_spread_capstone/data/raw/nflverse_schedules_2010_2025.csv


In [ ]:
list(raw_dir.iterdir())

[PosixPath('/content/nfl_spread_capstone/data/raw/nflverse_schedules_2010_2025.csv')]

In [ ]:
# Check the seasons included
schedules_2010_2025["season"].value_counts().sort_index()

,count
season,
2010,256
2011,256
2012,256
2013,256
2014,256
2015,256
2016,256
2017,256
2018,256


In [ ]:
# Preview important columns
important_cols = [
    "game_id",
    "season",
    "game_type",
    "week",
    "weekday",
    "gameday",
    "gametime",
    "away_team",
    "away_score",
    "home_team",
    "home_score",
    "location",
    "result",
    "total",
    "spread_line",
    "away_spread_odds",
    "home_spread_odds",
    "total_line",
    "under_odds",
    "over_odds",
    "home_moneyline",
    "away_moneyline",
    "div_game",
    "home_rest",
    "away_rest",
    "roof",
    "surface",
    "temp",
    "wind",
    "stadium",
    "away_coach",
    "home_coach",
    "referee"
]

schedules_2010_2025[important_cols].head(10)

,game_id,season,game_type,week,weekday,gameday,gametime,away_team,away_score,home_team,home_score,location,result,total,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,home_moneyline,away_moneyline,div_game,home_rest,away_rest,roof,surface,temp,wind,stadium,away_coach,home_coach,referee
2913,2010_01_MIN_NO,2010,REG,1,Thursday,2010-09-09,20:30,MIN,9.000,NO,14.000,Home,5.000,23.000,4.500,-105.000,-103.000,48.500,-104.000,-106.000,-220.000,197.000,0,7,7,dome,sportturf,NaN,NaN,Louisiana Superdome,Brad Childress,Sean Payton,Terry McAulay
2914,2010_01_MIA_BUF,2010,REG,1,Sunday,2010-09-12,13:00,MIA,15.000,BUF,10.000,Home,-5.000,25.000,-3.000,-106.000,-102.000,39.500,-110.000,100.000,140.000,-155.000,1,7,7,outdoors,astroplay,62.000,7.000,Ralph Wilson Stadium,Tony Sparano,Chan Gailey,Clete Blakeman
2915,2010_01_DET_CHI,2010,REG,1,Sunday,2010-09-12,13:00,DET,14.000,CHI,19.000,Home,5.000,33.000,6.500,103.000,-111.000,44.500,-105.000,-105.000,-280.000,248.000,1,7,7,outdoors,grass,75.000,10.000,Soldier Field,Jim Schwartz,Lovie Smith,Gene Steratore
2916,2010_01_IND_HOU,2010,REG,1,Sunday,2010-09-12,13:00,IND,24.000,HOU,34.000,Home,10.000,58.000,-1.000,-110.000,102.000,47.500,-102.000,-108.000,106.000,-117.000,1,7,7,closed,grass,NaN,NaN,Reliant Stadium,Jim Caldwell,Gary Kubiak,Ed Hochuli
2917,2010_01_DEN_JAX,2010,REG,1,Sunday,2010-09-12,13:00,DEN,17.000,JAX,24.000,Home,7.000,41.000,3.000,109.000,-118.000,41.500,-110.000,100.000,-185.000,166.000,0,7,7,outdoors,grass,90.000,10.000,EverBank Field,Josh McDaniels,Jack Del Rio,Walt Coleman
2918,2010_01_CIN_NE,2010,REG,1,Sunday,2010-09-12,13:00,CIN,24.000,NE,38.000,Home,14.000,62.000,5.000,-104.000,-104.000,44.500,-113.000,102.000,-230.000,205.000,0,7,7,outdoors,fieldturf,62.000,10.000,Gillette Stadium,Marvin Lewis,Bill Belichick,Carl Cheffers
2919,2010_01_CAR_NYG,2010,REG,1,Sunday,2010-09-12,13:00,CAR,18.000,NYG,31.000,Home,13.000,49.000,5.500,-107.000,-101.000,40.500,-107.000,-103.000,-240.000,214.000,0,7,7,outdoors,fieldturf,64.000,10.000,New Meadowlands Stadium,John Fox,Tom Coughlin,Jerome Boger
2920,2010_01_ATL_PIT,2010,REG,1,Sunday,2010-09-12,13:00,ATL,9.000,PIT,15.000,Home,6.000,24.000,-1.000,-106.000,-102.000,38.500,-108.000,-102.000,102.000,-112.000,0,7,7,outdoors,grass,65.000,9.000,Heinz Field,Mike Smith,Mike Tomlin,Scott Green
2921,2010_01_CLE_TB,2010,REG,1,Sunday,2010-09-12,13:00,CLE,14.000,TB,17.000,Home,3.000,31.000,2.500,102.000,-110.000,37.000,-106.000,-104.000,-135.000,122.000,0,7,7,outdoors,grass,87.000,6.000,Raymond James Stadium,Eric Mangini,Raheem Morris,Jeff Triplette
2922,2010_01_OAK_TEN,2010,REG,1,Sunday,2010-09-12,13:00,OAK,13.000,TEN,38.000,Home,25.000,51.000,6.500,-102.000,-106.000,39.500,-105.000,-105.000,-265.000,235.000,0,7,7,outdoors,grass,77.000,9.000,LP Field,Tom Cable,Jeff Fisher,Bill Leavy


In [ ]:
games = schedules_2010_2025[
    (schedules["home_score"].notna()) &
    (schedules["away_score"].notna()) &
    (schedules["spread_line"].notna())
].copy()

games.shape

/tmp/ipykernel_760/2149074349.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  games = schedules_2010_2025[


(4175, 46)

In [ ]:
games[important_cols].head()

,game_id,season,game_type,week,weekday,gameday,gametime,away_team,away_score,home_team,home_score,location,result,total,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,home_moneyline,away_moneyline,div_game,home_rest,away_rest,roof,surface,temp,wind,stadium,away_coach,home_coach,referee
2913,2010_01_MIN_NO,2010,REG,1,Thursday,2010-09-09,20:30,MIN,9.000,NO,14.000,Home,5.000,23.000,4.500,-105.000,-103.000,48.500,-104.000,-106.000,-220.000,197.000,0,7,7,dome,sportturf,NaN,NaN,Louisiana Superdome,Brad Childress,Sean Payton,Terry McAulay
2914,2010_01_MIA_BUF,2010,REG,1,Sunday,2010-09-12,13:00,MIA,15.000,BUF,10.000,Home,-5.000,25.000,-3.000,-106.000,-102.000,39.500,-110.000,100.000,140.000,-155.000,1,7,7,outdoors,astroplay,62.000,7.000,Ralph Wilson Stadium,Tony Sparano,Chan Gailey,Clete Blakeman
2915,2010_01_DET_CHI,2010,REG,1,Sunday,2010-09-12,13:00,DET,14.000,CHI,19.000,Home,5.000,33.000,6.500,103.000,-111.000,44.500,-105.000,-105.000,-280.000,248.000,1,7,7,outdoors,grass,75.000,10.000,Soldier Field,Jim Schwartz,Lovie Smith,Gene Steratore
2916,2010_01_IND_HOU,2010,REG,1,Sunday,2010-09-12,13:00,IND,24.000,HOU,34.000,Home,10.000,58.000,-1.000,-110.000,102.000,47.500,-102.000,-108.000,106.000,-117.000,1,7,7,closed,grass,NaN,NaN,Reliant Stadium,Jim Caldwell,Gary Kubiak,Ed Hochuli
2917,2010_01_DEN_JAX,2010,REG,1,Sunday,2010-09-12,13:00,DEN,17.000,JAX,24.000,Home,7.000,41.000,3.000,109.000,-118.000,41.500,-110.000,100.000,-185.000,166.000,0,7,7,outdoors,grass,90.000,10.000,EverBank Field,Josh McDaniels,Jack Del Rio,Walt Coleman


In [ ]:
# This will be positive if the home team won and nagative if the away team won

games["home_margin"] = games["home_score"] - games["away_score"]

games[[
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "result",
    "home_margin"
]].head(10)

,game_id,season,week,home_team,away_team,home_score,away_score,result,home_margin
2913,2010_01_MIN_NO,2010,1,NO,MIN,14.000,9.000,5.000,5.000
2914,2010_01_MIA_BUF,2010,1,BUF,MIA,10.000,15.000,-5.000,-5.000
2915,2010_01_DET_CHI,2010,1,CHI,DET,19.000,14.000,5.000,5.000
2916,2010_01_IND_HOU,2010,1,HOU,IND,34.000,24.000,10.000,10.000
2917,2010_01_DEN_JAX,2010,1,JAX,DEN,24.000,17.000,7.000,7.000
2918,2010_01_CIN_NE,2010,1,NE,CIN,38.000,24.000,14.000,14.000
2919,2010_01_CAR_NYG,2010,1,NYG,CAR,31.000,18.000,13.000,13.000
2920,2010_01_ATL_PIT,2010,1,PIT,ATL,15.000,9.000,6.000,6.000
2921,2010_01_CLE_TB,2010,1,TB,CLE,17.000,14.000,3.000,3.000
2922,2010_01_OAK_TEN,2010,1,TEN,OAK,38.000,13.000,25.000,25.000


In [ ]:
(games["result"] == games["home_margin"]).value_counts()

,count
True,4175


In [ ]:
# If the home team is -9 favorite and the home margin is 10 then the team covers
# 5 point home win & -3 home spread will give a result of 2

games["home_margin"] = games["home_score"] - games["away_score"]

# In this nflverse dataset, spread_line appears to align with home-team expected margin
games["market_home_margin"] = games["spread_line"]

# The home spread from a betting-ticket perspective is the opposite
games["home_spread"] = -games["market_home_margin"]

# Home team covers if actual margin is greater than market expected margin
games["spread_result"] = games["home_margin"] - games["market_home_margin"]

games["home_cover"] = np.where(games["spread_result"] > 0, 1, 0)

games["push"] = np.where(games["spread_result"] == 0, 1, 0)

games[[
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "result",
    "home_margin",
    "home_spread",
    "spread_result",
    "home_cover",
    "push"
]].head(15)

,game_id,season,week,home_team,away_team,home_score,away_score,result,home_margin,home_spread,spread_result,home_cover,push
2913,2010_01_MIN_NO,2010,1,NO,MIN,14.000,9.000,5.000,5.000,-4.500,0.500,1,0
2914,2010_01_MIA_BUF,2010,1,BUF,MIA,10.000,15.000,-5.000,-5.000,3.000,-2.000,0,0
2915,2010_01_DET_CHI,2010,1,CHI,DET,19.000,14.000,5.000,5.000,-6.500,-1.500,0,0
2916,2010_01_IND_HOU,2010,1,HOU,IND,34.000,24.000,10.000,10.000,1.000,11.000,1,0
2917,2010_01_DEN_JAX,2010,1,JAX,DEN,24.000,17.000,7.000,7.000,-3.000,4.000,1,0
2918,2010_01_CIN_NE,2010,1,NE,CIN,38.000,24.000,14.000,14.000,-5.000,9.000,1,0
2919,2010_01_CAR_NYG,2010,1,NYG,CAR,31.000,18.000,13.000,13.000,-5.500,7.500,1,0
2920,2010_01_ATL_PIT,2010,1,PIT,ATL,15.000,9.000,6.000,6.000,1.000,7.000,1,0
2921,2010_01_CLE_TB,2010,1,TB,CLE,17.000,14.000,3.000,3.000,-2.500,0.500,1,0
2922,2010_01_OAK_TEN,2010,1,TEN,OAK,38.000,13.000,25.000,25.000,-6.500,18.500,1,0


In [ ]:
(games["result"] == games["home_margin"]).value_counts()

,count
True,4175


In [ ]:
games[["home_margin", "home_spread",  "spread_result"]].describe()

,home_margin,home_spread,spread_result
count,4175.000,4175.000,4175.000
mean,2.015,-1.954,0.061
std,14.498,5.974,13.060
min,-49.000,-27.000,-52.000
25%,-7.000,-6.000,-8.000
50%,3.000,-3.000,0.000
75%,10.000,3.000,8.000
max,58.000,18.000,47.500


In [ ]:
print("Home cover rate excluding pushes:")
print(games.loc[games["push"] == 0, "home_cover"].mean())

print("Push rate:")
print(games["push"].mean())

Home cover rate excluding pushes:
0.4905382157778324
Push rate:
0.025389221556886228


In [ ]:
games_file = processed_dir / "nflverse_games_cleaned_2010_2025.csv"

games.to_csv(games_file, index=False)

print(f"Saved cleaned game-level data to: {games_file}")

Saved cleaned game-level data to: /content/nfl_spread_capstone/data/processed/nflverse_games_cleaned_2010_2025.csv


In [ ]:
list(processed_dir.iterdir())

[PosixPath('/content/nfl_spread_capstone/data/processed/nflverse_games_cleaned_2010_2025.csv')]

In [ ]:
pbp_seasons = list(range(2010, 2026))

pbp_list = []

for season in pbp_seasons:
    url = f"https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{season}.parquet"
    print(f"Loading {season}...")

    season_pbp = pd.read_parquet(url)
    season_pbp["season"] = season

    pbp_list.append(season_pbp)

pbp = pd.concat(pbp_list, ignore_index=True)

pbp.shape

Loading 2010...
Loading 2011...
Loading 2012...
Loading 2013...
Loading 2014...
Loading 2015...
Loading 2016...
Loading 2017...
Loading 2018...
Loading 2019...
Loading 2020...
Loading 2021...
Loading 2022...
Loading 2023...
Loading 2024...
Loading 2025...


(770337, 372)

In [ ]:
pbp.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,side_of_field,yardline_100,game_date,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,game_half,quarter_end,drive,sp,qtr,down,goal_to_go,time,yrdln,ydstogo,ydsnet,desc,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,pass_location,air_yards,yards_after_catch,run_location,run_gap,field_goal_result,kick_distance,extra_point_result,two_point_conv_result,home_timeouts_remaining,away_timeouts_remaining,timeout,timeout_team,td_team,td_player_name,td_player_id,posteam_timeouts_remaining,defteam_timeouts_remaining,total_home_score,total_away_score,posteam_score,defteam_score,score_differential,posteam_score_post,defteam_score_post,score_differential_post,no_score_prob,opp_fg_prob,opp_safety_prob,opp_td_prob,fg_prob,safety_prob,td_prob,extra_point_prob,two_point_conversion_prob,ep,epa,total_home_epa,total_away_epa,total_home_rush_epa,total_away_rush_epa,total_home_pass_epa,total_away_pass_epa,air_epa,yac_epa,comp_air_epa,comp_yac_epa,total_home_comp_air_epa,total_away_comp_air_epa,total_home_comp_yac_epa,total_away_comp_yac_epa,total_home_raw_air_epa,total_away_raw_air_epa,total_home_raw_yac_epa,total_away_raw_yac_epa,wp,def_wp,home_wp,away_wp,wpa,vegas_wpa,vegas_home_wpa,home_wp_post,away_wp_post,vegas_wp,vegas_home_wp,total_home_rush_wpa,total_away_rush_wpa,total_home_pass_wpa,total_away_pass_wpa,air_wpa,yac_wpa,comp_air_wpa,comp_yac_wpa,total_home_comp_air_wpa,total_away_comp_air_wpa,total_home_comp_yac_wpa,total_away_comp_yac_wpa,total_home_raw_air_wpa,total_away_raw_air_wpa,total_home_raw_yac_wpa,total_away_raw_yac_wpa,punt_blocked,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,fourth_down_converted,fourth_down_failed,incomplete_pass,touchback,interception,punt_inside_twenty,punt_in_endzone,punt_out_of_bounds,punt_downed,punt_fair_catch,kickoff_inside_twenty,kickoff_in_endzone,kickoff_out_of_bounds,kickoff_downed,kickoff_fair_catch,fumble_forced,fumble_not_forced,fumble_out_of_bounds,solo_tackle,safety,penalty,tackled_for_loss,fumble_lost,own_kickoff_recovery,own_kickoff_recovery_td,qb_hit,rush_attempt,pass_attempt,sack,touchdown,pass_touchdown,rush_touchdown,return_touchdown,extra_point_attempt,two_point_attempt,field_goal_attempt,kickoff_attempt,punt_attempt,fumble,complete_pass,assist_tackle,lateral_reception,lateral_rush,lateral_return,lateral_recovery,passer_player_id,passer_player_name,passing_yards,receiver_player_id,receiver_player_name,receiving_yards,rusher_player_id,rusher_player_name,rushing_yards,lateral_receiver_player_id,lateral_receiver_player_name,lateral_receiving_yards,lateral_rusher_player_id,lateral_rusher_player_name,lateral_rushing_yards,lateral_sack_player_id,lateral_sack_player_name,interception_player_id,interception_player_name,lateral_interception_player_id,lateral_interception_player_name,punt_returner_player_id,punt_returner_player_name,lateral_punt_returner_player_id,lateral_punt_returner_player_name,kickoff_returner_player_name,kickoff_returner_player_id,lateral_kickoff_returner_player_id,lateral_kickoff_returner_player_name,punter_player_id,punter_player_name,kicker_player_name,kicker_player_id,own_kickoff_recovery_player_id,own_kickoff_recovery_player_name,blocked_player_id,blocked_player_name,tackle_for_loss_1_player_id,tackle_for_loss_1_player_name,tackle_for_loss_2_player_id,tackle_for_loss_2_player_name,qb_hit_1_player_id,qb_hit_1_player_name,qb_hit_2_player_id,qb_hit_2_player_name,forced_fumble_player_1_team,forced_fumble_player_1_player_id,forced_fumble_player_1_player_name,forced_fumble_player_2_team,forced_fumble_player_2_player_id,forced_fumble_player_2_player_name,solo_tackle_1_team,solo_tackle_2_team,solo_tackle_1_player_id,solo_tackle_2_player_id,solo_tackle_1_player_name,solo_tackle_2_player_name,assist_tackle_1_player_id,assist_tackle_1_player_name,assist_tackle_1_team,assist_tack

In [ ]:
pbp.columns.tolist()

['play_id',
 'game_id',
 'old_game_id',
 'home_team',
 'away_team',
 'season_type',
 'week',
 'posteam',
 'posteam_type',
 'defteam',
 'side_of_field',
 'yardline_100',
 'game_date',
 'quarter_seconds_remaining',
 'half_seconds_remaining',
 'game_seconds_remaining',
 'game_half',
 'quarter_end',
 'drive',
 'sp',
 'qtr',
 'down',
 'goal_to_go',
 'time',
 'yrdln',
 'ydstogo',
 'ydsnet',
 'desc',
 'play_type',
 'yards_gained',
 'shotgun',
 'no_huddle',
 'qb_dropback',
 'qb_kneel',
 'qb_spike',
 'qb_scramble',
 'pass_length',
 'pass_location',
 'air_yards',
 'yards_after_catch',
 'run_location',
 'run_gap',
 'field_goal_result',
 'kick_distance',
 'extra_point_result',
 'two_point_conv_result',
 'home_timeouts_remaining',
 'away_timeouts_remaining',
 'timeout',
 'timeout_team',
 'td_team',
 'td_player_name',
 'td_player_id',
 'posteam_timeouts_remaining',
 'defteam_timeouts_remaining',
 'total_home_score',
 'total_away_score',
 'posteam_score',
 'defteam_score',
 'score_differential',
 'po

In [ ]:
pbp_cols = [
    # Game/team identifiers
    "game_id",
    "season",
    "season_type",
    "week",
    "game_date",
    "home_team",
    "away_team",
    "posteam",
    "posteam_type",
    "defteam",

    # Game context
    "qtr",
    "down",
    "ydstogo",
    "yardline_100",
    "game_seconds_remaining",
    "score_differential",
    "roof",
    "surface",
    "temp",
    "wind",
    "div_game",
    "spread_line",
    "total_line",

    # Play type and tendencies
    "play_type",
    "qb_dropback",
    "pass_attempt",
    "rush_attempt",
    "pass",
    "rush",
    "shotgun",
    "no_huddle",
    "qb_scramble",

    # Efficiency / production
    "yards_gained",
    "air_yards",
    "yards_after_catch",
    "passing_yards",
    "rushing_yards",
    "receiving_yards",

    # EPA / WP
    "ep",
    "epa",
    "qb_epa",
    "air_epa",
    "yac_epa",
    "wp",
    "wpa",
    "vegas_wpa",

    # Success / expected passing
    "success",
    "first_down",
    "series_success",
    "cp",
    "cpoe",
    "xpass",
    "pass_oe",

    # Expected yards after catch
    "xyac_epa",
    "xyac_mean_yardage",
    "xyac_median_yardage",
    "xyac_success",
    "xyac_fd",

    # Pressure / negative plays
    "sack",
    "qb_hit",
    "tackled_for_loss",
    "interception",
    "fumble",
    "fumble_lost",

    # Situational conversions
    "third_down_converted",
    "third_down_failed",
    "fourth_down_converted",
    "fourth_down_failed",

    # Scoring
    "touchdown",
    "pass_touchdown",
    "rush_touchdown"
]

existing_pbp_cols = [col for col in pbp_cols if col in pbp.columns]
missing_pbp_cols = [col for col in pbp_cols if col not in pbp.columns]

print("Existing columns:")
print(existing_pbp_cols)

print("\nMissing columns:")
print(missing_pbp_cols)

Existing columns:
['game_id', 'season', 'season_type', 'week', 'game_date', 'home_team', 'away_team', 'posteam', 'posteam_type', 'defteam', 'qtr', 'down', 'ydstogo', 'yardline_100', 'game_seconds_remaining', 'score_differential', 'roof', 'surface', 'temp', 'wind', 'div_game', 'spread_line', 'total_line', 'play_type', 'qb_dropback', 'pass_attempt', 'rush_attempt', 'pass', 'rush', 'shotgun', 'no_huddle', 'qb_scramble', 'yards_gained', 'air_yards', 'yards_after_catch', 'passing_yards', 'rushing_yards', 'receiving_yards', 'ep', 'epa', 'qb_epa', 'air_epa', 'yac_epa', 'wp', 'wpa', 'vegas_wpa', 'success', 'first_down', 'series_success', 'cp', 'cpoe', 'xpass', 'pass_oe', 'xyac_epa', 'xyac_mean_yardage', 'xyac_median_yardage', 'xyac_success', 'xyac_fd', 'sack', 'qb_hit', 'tackled_for_loss', 'interception', 'fumble', 'fumble_lost', 'third_down_converted', 'third_down_failed', 'fourth_down_converted', 'fourth_down_failed', 'touchdown', 'pass_touchdown', 'rush_touchdown']

Missing columns:
[]


In [ ]:
pbp_small = pbp[existing_pbp_cols].copy()

pbp_small.shape

(770337, 71)

In [ ]:
pbp_clean = pbp_small[
    (pbp_small["posteam"].notna()) &
    (pbp_small["defteam"].notna()) &
    (pbp_small["epa"].notna()) &
    (pbp_small["season_type"] == "REG")
].copy()

pbp_clean.shape

(694314, 71)

In [ ]:
pbp_clean["play_type"].value_counts(dropna=False).head(20)

,count
play_type,
pass,306231
run,217638
kickoff,42582
no_play,39283
punt,36351
extra_point,19604
field_goal,16240
None,8945
qb_kneel,6331


In [ ]:
offensive_plays = pbp_clean[
    pbp_clean["play_type"].isin(["pass", "run"])
].copy()

offensive_plays.shape

(523869, 71)

In [ ]:
# Early downs: 1st and 2nd down
offensive_plays["early_down"] = offensive_plays["down"].isin([1, 2]).astype(int)

# Early-down pass
offensive_plays["early_down_pass"] = (
    (offensive_plays["early_down"] == 1) &
    (offensive_plays["pass_attempt"] == 1)
).astype(int)

# Early-down rush
offensive_plays["early_down_rush"] = (
    (offensive_plays["early_down"] == 1) &
    (offensive_plays["rush_attempt"] == 1)
).astype(int)

# Successful early-down pass
offensive_plays["early_down_pass_success"] = (
    (offensive_plays["early_down_pass"] == 1) &
    (offensive_plays["success"] == 1)
).astype(int)

# Pressure proxy: sack or QB hit
offensive_plays["pressure_allowed"] = (
    (offensive_plays["sack"] == 1) |
    (offensive_plays["qb_hit"] == 1)
).astype(int)

# Defensive pressure created, from defense perspective later
offensive_plays["pressure_created"] = offensive_plays["pressure_allowed"]

# Explosive plays
# Common football definitions: pass gain >= 20 yards, rush gain >= 10 yards
offensive_plays["explosive_pass"] = (
    (offensive_plays["pass_attempt"] == 1) &
    (offensive_plays["yards_gained"] >= 20)
).astype(int)

offensive_plays["explosive_rush"] = (
    (offensive_plays["rush_attempt"] == 1) &
    (offensive_plays["yards_gained"] >= 10)
).astype(int)

offensive_plays["explosive_play"] = (
    (offensive_plays["explosive_pass"] == 1) |
    (offensive_plays["explosive_rush"] == 1)
).astype(int)

# Turnover play
offensive_plays["turnover"] = (
    (offensive_plays["interception"] == 1) |
    (offensive_plays["fumble_lost"] == 1)
).astype(int)

# Negative play
offensive_plays["negative_play"] = (
    (offensive_plays["yards_gained"] < 0) |
    (offensive_plays["sack"] == 1) |
    (offensive_plays["tackled_for_loss"] == 1)
).astype(int)

# Third/fourth down attempts
offensive_plays["third_down_attempt"] = (
    (offensive_plays["third_down_converted"] == 1) |
    (offensive_plays["third_down_failed"] == 1)
).astype(int)

offensive_plays["fourth_down_attempt"] = (
    (offensive_plays["fourth_down_converted"] == 1) |
    (offensive_plays["fourth_down_failed"] == 1)
).astype(int)

offensive_plays.head()

,game_id,season,season_type,week,game_date,home_team,away_team,posteam,posteam_type,defteam,qtr,down,ydstogo,yardline_100,game_seconds_remaining,score_differential,roof,surface,temp,wind,div_game,spread_line,total_line,play_type,qb_dropback,pass_attempt,rush_attempt,pass,rush,shotgun,no_huddle,qb_scramble,yards_gained,air_yards,yards_after_catch,passing_yards,rushing_yards,receiving_yards,ep,epa,qb_epa,air_epa,yac_epa,wp,wpa,vegas_wpa,success,first_down,series_success,cp,cpoe,xpass,pass_oe,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,sack,qb_hit,tackled_for_loss,interception,fumble,fumble_lost,third_down_converted,third_down_failed,fourth_down_converted,fourth_down_failed,touchdown,pass_touchdown,rush_touchdown,early_down,early_down_pass,early_down_rush,early_down_pass_success,pressure_allowed,pressure_created,explosive_pass,explosive_rush,explosive_play,turnover,negative_play,third_down_attempt,fourth_down_attempt
2,2010_01_ARI_STL,2010,REG,1,2010-09-12,LA,ARI,ARI,away,LA,1.000,1.000,10.000,78.000,3595.000,0.000,dome,astroplay,NaN,NaN,1,-3.000,39.500,pass,1.000,1.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,NaN,0.000,0.305,-0.565,-0.565,-0.565,0.000,0.434,-0.018,-0.016,0.000,0.000,1.000,0.708,29.197,0.502,49.797,0.900,6.992,6.000,0.691,0.224,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1,1,0,0,0,0,0,0,0,0,0,0,0
3,2010_01_ARI_STL,2010,REG,1,2010-09-12,LA,ARI,ARI,away,LA,1.000,2.000,10.000,78.000,3564.000,0.000,dome,astroplay,NaN,NaN,1,-3.000,39.500,run,0.000,0.000,1.000,0.000,1.000,0.000,0.000,0.000,5.000,NaN,NaN,NaN,5.000,NaN,-0.259,-0.022,-0.022,NaN,NaN,0.416,-0.013,-0.004,0.000,0.000,1.000,NaN,NaN,0.500,-49.982,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1,0,1,0,0,0,0,0,0,0,0,0,0
4,2010_01_ARI_STL,2010,REG,1,2010-09-12,LA,ARI,ARI,away,LA,1.000,3.000,5.000,73.000,3523.000,0.000,dome,astroplay,NaN,NaN,1,-3.000,39.500,pass,1.000,1.000,0.000,1.000,0.000,1.000,0.000,0.000,18.000,7.000,11.000,18.000,NaN,18.000,-0.282,2.208,2.208,1.449,0.759,0.403,0.059,0.048,1.000,1.000,1.000,0.640,36.021,0.963,3.713,0.226,3.391,1.000,0.998,0.998,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0,0,0,0,0,0,0,0,0,0,0,1,0
5,2010_01_ARI_STL,2010,REG,1,2010-09-12,LA,ARI,ARI,away,LA,1.000,1.000,10.000,55.000,3497.000,0.000,dome,astroplay,NaN,NaN,1,-3.000,39.500,pass,1.000,1.000,0.000,1.000,0.000,1.000,0.000,0.000,17.000,0.000,17.000,17.000,NaN,17.000,1.926,1.344,1.344,-0.472,1.816,0.462,0.068,0.034,1.000,1.000,1.000,0.703,29.656,0.477,52.268,0.853,7.082,6.000,0.692,0.233,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1,1,0,1,0,0,0,0,0,0,0,0,0
6,2010_01_ARI_STL,2010,REG,1,2010-09-12,LA,ARI,ARI,away,LA,1.000,1.000,10.000,38.000,3472.000,0.000,dome,astroplay,NaN,NaN,1,-3.000,39.500,run,0.000,0.000,1.000,0.000,1.000,0.000,0.000,0.000,2.000,NaN,NaN,NaN,2.000,NaN,3.270,-0.344,-0.344,NaN,NaN,0.530,-0.016,-0.020,0.000,0.000,1.000,NaN,NaN,0.473,-47.311,NaN,NaN,NaN,NaN,NaN,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1,0,1,0,0,0,0,0,0,0,0,0,0


In [ ]:
team_game_offense = (
    offensive_plays
    .groupby(["game_id", "season", "week", "game_date", "posteam"])
    .agg(
        offensive_plays=("epa", "count"),

        # Overall efficiency
        off_epa_per_play=("epa", "mean"),
        off_success_rate=("success", "mean"),
        yards_per_play=("yards_gained", "mean"),

        # Passing efficiency
        pass_rate=("pass_attempt", "mean"),
        pass_epa_per_play=("epa", lambda x: x[offensive_plays.loc[x.index, "pass_attempt"] == 1].mean()),
        pass_success_rate=("success", lambda x: x[offensive_plays.loc[x.index, "pass_attempt"] == 1].mean()),
        yards_per_pass=("yards_gained", lambda x: x[offensive_plays.loc[x.index, "pass_attempt"] == 1].mean()),
        avg_air_yards=("air_yards", "mean"),
        avg_yac=("yards_after_catch", "mean"),
        avg_cpoe=("cpoe", "mean"),
        avg_xpass=("xpass", "mean"),
        avg_pass_oe=("pass_oe", "mean"),

        # Rushing efficiency
        rush_rate=("rush_attempt", "mean"),
        rush_epa_per_play=("epa", lambda x: x[offensive_plays.loc[x.index, "rush_attempt"] == 1].mean()),
        rush_success_rate=("success", lambda x: x[offensive_plays.loc[x.index, "rush_attempt"] == 1].mean()),
        yards_per_rush=("yards_gained", lambda x: x[offensive_plays.loc[x.index, "rush_attempt"] == 1].mean()),

        # Early-down passing
        early_down_pass_rate=("early_down_pass", "mean"),
        early_down_pass_success_rate=("early_down_pass_success", "mean"),
        early_down_epa_per_play=("epa", lambda x: x[offensive_plays.loc[x.index, "early_down"] == 1].mean()),
        early_down_pass_epa=("epa", lambda x: x[offensive_plays.loc[x.index, "early_down_pass"] == 1].mean()),

        # Pressure / negative plays
        pressure_allowed_rate=("pressure_allowed", "mean"),
        sack_rate=("sack", "mean"),
        qb_hit_rate_allowed=("qb_hit", "mean"),
        negative_play_rate=("negative_play", "mean"),
        turnover_rate=("turnover", "mean"),

        # Explosiveness
        explosive_play_rate=("explosive_play", "mean"),
        explosive_pass_rate=("explosive_pass", "mean"),
        explosive_rush_rate=("explosive_rush", "mean"),

        # Situational
        third_down_attempts=("third_down_attempt", "sum"),
        third_down_conversions=("third_down_converted", "sum"),
        fourth_down_attempts=("fourth_down_attempt", "sum"),
        fourth_down_conversions=("fourth_down_converted", "sum"),

        # Expected YAC
        avg_xyac_epa=("xyac_epa", "mean"),
        avg_xyac_yards=("xyac_mean_yardage", "mean"),
        avg_xyac_success=("xyac_success", "mean")
    )
    .reset_index()
    .rename(columns={"posteam": "team"})
)

team_game_offense.head()

,game_id,season,week,game_date,team,offensive_plays,off_epa_per_play,off_success_rate,yards_per_play,pass_rate,pass_epa_per_play,pass_success_rate,yards_per_pass,avg_air_yards,avg_yac,avg_cpoe,avg_xpass,avg_pass_oe,rush_rate,rush_epa_per_play,rush_success_rate,yards_per_rush,early_down_pass_rate,early_down_pass_success_rate,early_down_epa_per_play,early_down_pass_epa,pressure_allowed_rate,sack_rate,qb_hit_rate_allowed,negative_play_rate,turnover_rate,explosive_play_rate,explosive_pass_rate,explosive_rush_rate,third_down_attempts,third_down_conversions,fourth_down_attempts,fourth_down_conversions,avg_xyac_epa,avg_xyac_yards,avg_xyac_success
0,2010_01_ARI_STL,2010,1,2010-09-12,ARI,64,-0.123,0.359,5.906,0.672,0.066,0.372,6.186,10.220,5.955,0.791,0.637,5.017,0.328,-0.509,0.333,5.333,0.469,0.172,-0.176,0.057,0.156,0.031,0.141,0.109,0.062,0.156,0.094,0.062,13,5.000,0,0.000,0.610,4.987,0.804
1,2010_01_ARI_STL,2010,1,2010-09-12,LA,81,-0.201,0.358,4.012,0.704,-0.242,0.351,4.211,6.855,3.312,-6.194,0.705,-0.208,0.296,-0.102,0.375,3.542,0.457,0.160,-0.275,-0.320,0.086,0.025,0.086,0.074,0.037,0.062,0.037,0.025,20,8.000,3,2.000,0.760,4.858,0.714
2,2010_01_ATL_PIT,2010,1,2010-09-12,ATL,71,-0.105,0.451,4.155,0.648,-0.049,0.457,5.152,8.386,1.963,-1.153,0.594,5.422,0.352,-0.209,0.440,2.320,0.479,0.254,-0.004,0.117,0.042,0.028,0.042,0.085,0.014,0.028,0.028,0.000,16,6.000,0,0.000,0.565,4.483,0.839
3,2010_01_ATL_PIT,2010,1,2010-09-12,PIT,60,-0.028,0.367,5.900,0.483,0.010,0.448,7.276,8.038,4.611,4.465,0.577,-6.005,0.517,-0.064,0.290,4.613,0.333,0.167,0.040,0.101,0.050,0.050,0.050,0.083,0.017,0.100,0.067,0.033,14,4.000,0,0.000,0.599,5.570,0.812
4,2010_01_BAL_NYJ,2010,1,2010-09-13,BAL,74,-0.261,0.351,3.824,0.541,-0.120,0.400,5.825,10.526,3.200,-2.265,0.545,-0.415,0.459,-0.426,0.294,1.471,0.338,0.095,-0.545,-0.554,0.068,0.027,0.054,0.122,0.041,0.054,0.054,0.000,19,11.000,0,0.000,0.811,5.915,0.849


In [ ]:
team_game_offense.shape

(8350, 41)

In [ ]:
team_game_offense["season"].value_counts().sort_index() / 2

,count
season,
2010,256.000
2011,256.000
2012,256.000
2013,256.000
2014,256.000
2015,256.000
2016,256.000
2017,256.000
2018,256.000


In [ ]:
team_game_offense["third_down_conversion_rate"] = (
    team_game_offense["third_down_conversions"] / team_game_offense["third_down_attempts"]
)

team_game_offense["fourth_down_conversion_rate"] = (
    team_game_offense["fourth_down_conversions"] / team_game_offense["fourth_down_attempts"]
)

team_game_offense[[
    "team",
    "season",
    "week",
    "off_epa_per_play",
    "off_success_rate",
    "pass_epa_per_play",
    "rush_epa_per_play",
    "early_down_pass_success_rate",
    "pressure_allowed_rate",
    "explosive_play_rate"
]].head()

,team,season,week,off_epa_per_play,off_success_rate,pass_epa_per_play,rush_epa_per_play,early_down_pass_success_rate,pressure_allowed_rate,explosive_play_rate
0,ARI,2010,1,-0.123,0.359,0.066,-0.509,0.172,0.156,0.156
1,LA,2010,1,-0.201,0.358,-0.242,-0.102,0.160,0.086,0.062
2,ATL,2010,1,-0.105,0.451,-0.049,-0.209,0.254,0.042,0.028
3,PIT,2010,1,-0.028,0.367,0.010,-0.064,0.167,0.050,0.100
4,BAL,2010,1,-0.261,0.351,-0.120,-0.426,0.095,0.068,0.054


In [ ]:
team_game_defense = (
    offensive_plays
    .groupby(["game_id", "season", "week", "game_date", "defteam"])
    .agg(
        defensive_plays=("epa", "count"),

        # Defensive efficiency allowed
        def_epa_allowed_per_play=("epa", "mean"),
        def_success_rate_allowed=("success", "mean"),
        def_yards_allowed_per_play=("yards_gained", "mean"),

        # Passing defense
        def_pass_epa_allowed=("epa", lambda x: x[offensive_plays.loc[x.index, "pass_attempt"] == 1].mean()),
        def_pass_success_allowed=("success", lambda x: x[offensive_plays.loc[x.index, "pass_attempt"] == 1].mean()),
        def_yards_allowed_per_pass=("yards_gained", lambda x: x[offensive_plays.loc[x.index, "pass_attempt"] == 1].mean()),

        # Rushing defense
        def_rush_epa_allowed=("epa", lambda x: x[offensive_plays.loc[x.index, "rush_attempt"] == 1].mean()),
        def_rush_success_allowed=("success", lambda x: x[offensive_plays.loc[x.index, "rush_attempt"] == 1].mean()),
        def_yards_allowed_per_rush=("yards_gained", lambda x: x[offensive_plays.loc[x.index, "rush_attempt"] == 1].mean()),

        # Early-down defense
        def_early_down_epa_allowed=("epa", lambda x: x[offensive_plays.loc[x.index, "early_down"] == 1].mean()),
        def_early_down_pass_epa_allowed=("epa", lambda x: x[offensive_plays.loc[x.index, "early_down_pass"] == 1].mean()),
        def_early_down_pass_success_allowed=("early_down_pass_success", "mean"),

        # Pressure created
        pressure_created_rate=("pressure_created", "mean"),
        sack_created_rate=("sack", "mean"),
        qb_hit_created_rate=("qb_hit", "mean"),
        tackle_for_loss_rate=("tackled_for_loss", "mean"),

        # Explosiveness allowed
        explosive_allowed_rate=("explosive_play", "mean"),
        explosive_pass_allowed_rate=("explosive_pass", "mean"),
        explosive_rush_allowed_rate=("explosive_rush", "mean"),

        # Turnovers forced
        turnover_forced_rate=("turnover", "mean")
    )
    .reset_index()
    .rename(columns={"defteam": "team"})
)

team_game_defense.head()

,game_id,season,week,game_date,team,defensive_plays,def_epa_allowed_per_play,def_success_rate_allowed,def_yards_allowed_per_play,def_pass_epa_allowed,def_pass_success_allowed,def_yards_allowed_per_pass,def_rush_epa_allowed,def_rush_success_allowed,def_yards_allowed_per_rush,def_early_down_epa_allowed,def_early_down_pass_epa_allowed,def_early_down_pass_success_allowed,pressure_created_rate,sack_created_rate,qb_hit_created_rate,tackle_for_loss_rate,explosive_allowed_rate,explosive_pass_allowed_rate,explosive_rush_allowed_rate,turnover_forced_rate
0,2010_01_ARI_STL,2010,1,2010-09-12,ARI,81,-0.201,0.358,4.012,-0.242,0.351,4.211,-0.102,0.375,3.542,-0.275,-0.320,0.160,0.086,0.025,0.086,0.025,0.062,0.037,0.025,0.037
1,2010_01_ARI_STL,2010,1,2010-09-12,LA,64,-0.123,0.359,5.906,0.066,0.372,6.186,-0.509,0.333,5.333,-0.176,0.057,0.172,0.156,0.031,0.141,0.078,0.156,0.094,0.062,0.062
2,2010_01_ATL_PIT,2010,1,2010-09-12,ATL,60,-0.028,0.367,5.900,0.010,0.448,7.276,-0.064,0.290,4.613,0.040,0.101,0.167,0.050,0.050,0.050,0.017,0.100,0.067,0.033,0.017
3,2010_01_ATL_PIT,2010,1,2010-09-12,PIT,71,-0.105,0.451,4.155,-0.049,0.457,5.152,-0.209,0.440,2.320,-0.004,0.117,0.254,0.042,0.028,0.042,0.056,0.028,0.028,0.000,0.014
4,2010_01_BAL_NYJ,2010,1,2010-09-13,BAL,43,-0.387,0.326,4.116,-0.551,0.261,2.609,-0.197,0.400,5.850,-0.043,-0.071,0.116,0.070,0.047,0.070,0.023,0.070,0.000,0.070,0.023


In [ ]:
team_game_defense.shape

(8350, 26)

In [ ]:
team_game_stats = team_game_offense.merge(
    team_game_defense,
    on=["game_id", "season", "week", "game_date", "team"],
    how="inner"
)

team_game_stats.shape

(8350, 64)

In [ ]:
team_game_stats.head()

,game_id,season,week,game_date,team,offensive_plays,off_epa_per_play,off_success_rate,yards_per_play,pass_rate,pass_epa_per_play,pass_success_rate,yards_per_pass,avg_air_yards,avg_yac,avg_cpoe,avg_xpass,avg_pass_oe,rush_rate,rush_epa_per_play,rush_success_rate,yards_per_rush,early_down_pass_rate,early_down_pass_success_rate,early_down_epa_per_play,early_down_pass_epa,pressure_allowed_rate,sack_rate,qb_hit_rate_allowed,negative_play_rate,turnover_rate,explosive_play_rate,explosive_pass_rate,explosive_rush_rate,third_down_attempts,third_down_conversions,fourth_down_attempts,fourth_down_conversions,avg_xyac_epa,avg_xyac_yards,avg_xyac_success,third_down_conversion_rate,fourth_down_conversion_rate,defensive_plays,def_epa_allowed_per_play,def_success_rate_allowed,def_yards_allowed_per_play,def_pass_epa_allowed,def_pass_success_allowed,def_yards_allowed_per_pass,def_rush_epa_allowed,def_rush_success_allowed,def_yards_allowed_per_rush,def_early_down_epa_allowed,def_early_down_pass_epa_allowed,def_early_down_pass_success_allowed,pressure_created_rate,sack_created_rate,qb_hit_created_rate,tackle_for_loss_rate,explosive_allowed_rate,explosive_pass_allowed_rate,explosive_rush_allowed_rate,turnover_forced_rate
0,2010_01_ARI_STL,2010,1,2010-09-12,ARI,64,-0.123,0.359,5.906,0.672,0.066,0.372,6.186,10.220,5.955,0.791,0.637,5.017,0.328,-0.509,0.333,5.333,0.469,0.172,-0.176,0.057,0.156,0.031,0.141,0.109,0.062,0.156,0.094,0.062,13,5.000,0,0.000,0.610,4.987,0.804,0.385,NaN,81,-0.201,0.358,4.012,-0.242,0.351,4.211,-0.102,0.375,3.542,-0.275,-0.320,0.160,0.086,0.025,0.086,0.025,0.062,0.037,0.025,0.037
1,2010_01_ARI_STL,2010,1,2010-09-12,LA,81,-0.201,0.358,4.012,0.704,-0.242,0.351,4.211,6.855,3.312,-6.194,0.705,-0.208,0.296,-0.102,0.375,3.542,0.457,0.160,-0.275,-0.320,0.086,0.025,0.086,0.074,0.037,0.062,0.037,0.025,20,8.000,3,2.000,0.760,4.858,0.714,0.400,0.667,64,-0.123,0.359,5.906,0.066,0.372,6.186,-0.509,0.333,5.333,-0.176,0.057,0.172,0.156,0.031,0.141,0.078,0.156,0.094,0.062,0.062
2,2010_01_ATL_PIT,2010,1,2010-09-12,ATL,71,-0.105,0.451,4.155,0.648,-0.049,0.457,5.152,8.386,1.963,-1.153,0.594,5.422,0.352,-0.209,0.440,2.320,0.479,0.254,-0.004,0.117,0.042,0.028,0.042,0.085,0.014,0.028,0.028,0.000,16,6.000,0,0.000,0.565,4.483,0.839,0.375,NaN,60,-0.028,0.367,5.900,0.010,0.448,7.276,-0.064,0.290,4.613,0.040,0.101,0.167,0.050,0.050,0.050,0.017,0.100,0.067,0.033,0.017
3,2010_01_ATL_PIT,2010,1,2010-09-12,PIT,60,-0.028,0.367,5.900,0.483,0.010,0.448,7.276,8.038,4.611,4.465,0.577,-6.005,0.517,-0.064,0.290,4.613,0.333,0.167,0.040,0.101,0.050,0.050,0.050,0.083,0.017,0.100,0.067,0.033,14,4.000,0,0.000,0.599,5.570,0.812,0.286,NaN,71,-0.105,0.451,4.155,-0.049,0.457,5.152,-0.209,0.440,2.320,-0.004,0.117,0.254,0.042,0.028,0.042,0.056,0.028,0.028,0.000,0.014
4,2010_01_BAL_NYJ,2010,1,2010-09-13,BAL,74,-0.261,0.351,3.824,0.541,-0.120,0.400,5.825,10.526,3.200,-2.265,0.545,-0.415,0.459,-0.426,0.294,1.471,0.338,0.095,-0.545,-0.554,0.068,0.027,0.054,0.122,0.041,0.054,0.054,0.000,19,11.000,0,0.000,0.811,5.915,0.849,0.579,NaN,43,-0.387,0.326,4.116,-0.551,0.261,2.609,-0.197,0.400,5.850,-0.043,-0.071,0.116,0.070,0.047,0.070,0.023,0.070,0.000,0.070,0.023


In [ ]:
team_game_stats_file = processed_dir / "team_game_advanced_stats_2010_2025.csv"


print(f"Saved team-game advanced stats to: {team_game_stats_file}")

Saved team-game advanced stats to: /content/nfl_spread_capstone/data/processed/team_game_advanced_stats_2010_2025.csv


# Proposal Notebook Summary

This notebook explored nflverse data for a proposed NFL spread prediction capstone project. The schedule dataset includes game-level information such as teams, scores, spread lines, totals, moneylines, rest days, weather, stadium, roof, and surface. The play-by-play dataset includes detailed play-level information that can be used to engineer advanced team performance features.

The main target variable for the future machine learning model will be `home_margin`, calculated as home score minus away score. The sportsbook spread will be converted into `market_home_margin`, which represents the market's expected home-team margin.

The project will later compare a model-predicted spread to the sportsbook spread. If the model's predicted margin differs from the sportsbook margin by more than a selected threshold, the game may be labeled as a potential simulated bet. Otherwise, the model will recommend no bet.

Advanced features identified in this notebook include offensive EPA per play, defensive EPA allowed per play, success rate, passing EPA, rushing EPA, early-down passing success, pressure rate, explosive play rate, turnover rate, and situational conversion rates.

The next stage of the project will be to create pregame rolling averages so that every model feature only uses information available before each game. This is necessary to avoid data leakage.

In [ ]:
print("Schedule Dataset")
print("----------------")
print(f"Rows: {schedules_2010_2025.shape[0]}")
print(f"Columns: {schedules_2010_2025.shape[1]}")

print("\nCompleted Regular Season Games Dataset")
print("--------------------------------------")
print(f"Rows: {games.shape[0]}")
print(f"Columns: {games.shape[1]}")

print("\nPlay-by-Play Dataset")
print("--------------------")
print(f"Rows: {pbp.shape[0]}")
print(f"Columns: {pbp.shape[1]}")

print("\nSelected Play-by-Play Dataset")
print("-----------------------------")
print(f"Rows: {pbp_small.shape[0]}")
print(f"Columns: {pbp_small.shape[1]}")

print("\nTeam-Game Advanced Stats Dataset")
print("--------------------------------")
print(f"Rows: {team_game_stats.shape[0]}")
print(f"Columns: {team_game_stats.shape[1]}")

print("\nSeasons Included")
print("----------------")
print(f"Start season: {schedules_2010_2025['season'].min()}")
print(f"End season: {schedules_2010_2025['season'].max()}")

Schedule Dataset
----------------
Rows: 4175
Columns: 46

Completed Regular Season Games Dataset
--------------------------------------
Rows: 4175
Columns: 52

Play-by-Play Dataset
--------------------
Rows: 770337
Columns: 372

Selected Play-by-Play Dataset
-----------------------------
Rows: 770337
Columns: 71

Team-Game Advanced Stats Dataset
--------------------------------
Rows: 8350
Columns: 64

Seasons Included
----------------
Start season: 2010
End season: 2025


# Assignment 03 - Complete EDA

The previous section loaded nflverse schedule, betting, and play-by-play data and created team-game advanced statistics. This section continues the project by preparing a tidy game-level dataset for exploratory data analysis.

The team-game advanced statistics dataset contains two rows per game: one for each team. For machine learning, the final dataset should contain one row per NFL game. To avoid data leakage, this section creates pregame rolling averages for each team. These rolling averages use only games played before the current game.

The final EDA dataset will include the target variable `home_margin`, sportsbook market variables, rest/weather/game context variables, and selected advanced team performance features.

In [ ]:
# Confirm the current datasets created in the previous section
print("Completed regular season games:", games.shape)
print("Team-game advanced stats:", team_game_stats.shape)

print("\nGames columns:")
print(games.columns.tolist())

print("\nTeam-game stats columns:")
print(team_game_stats.columns.tolist())

Completed regular season games: (4175, 52)
Team-game advanced stats: (8350, 64)

Games columns:
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium', 'home_margin', 'market_home_margin', 'home_spread', 'spread_result', 'home_cover', 'push']

Team-game stats columns:
['game_id', 'season', 'week', 'game_date', 'team', 'offensive_plays', 'off_epa_per_play', 'off_success_rate', 'yards_per_play', 'pass_rate', 'pass_epa_per_play', 'pass_success_rate', 'yards_per_pass', 'avg_air_yards

In [ ]:
# Convert game_date to datetime so games can be sorted chronologically
team_game_stats["game_date"] = pd.to_datetime(team_game_stats["game_date"])

# Sort by team, season, and game date before creating rolling averages
team_game_stats = team_game_stats.sort_values(
    ["team", "season", "game_date", "game_id"]
).copy()

team_game_stats[["team", "season", "week", "game_date"]].head()

,team,season,week,game_date
0,ARI,2010,1,2010-09-12
32,ARI,2010,2,2010-09-19
82,ARI,2010,3,2010-09-26
96,ARI,2010,4,2010-10-03
138,ARI,2010,5,2010-10-10


In [ ]:
# Automatically select all numeric team-level features from team_game_stats.
# This keeps the starting feature set broad so we can narrow it down later
# using EDA, correlation, missing values, and model performance.

identifier_cols = [
    "game_id",
    "season",
    "week",
    "game_date",
    "team"
]

# Select numeric columns only
numeric_team_cols = team_game_stats.select_dtypes(include=["number"]).columns.tolist()

# Remove identifiers that should not be treated as model features
selected_team_features = [
    col for col in numeric_team_cols
    if col not in identifier_cols
]

print("Number of selected team features:", len(selected_team_features))
selected_team_features

Number of selected team features: 59


['offensive_plays',
 'off_epa_per_play',
 'off_success_rate',
 'yards_per_play',
 'pass_rate',
 'pass_epa_per_play',
 'pass_success_rate',
 'yards_per_pass',
 'avg_air_yards',
 'avg_yac',
 'avg_cpoe',
 'avg_xpass',
 'avg_pass_oe',
 'rush_rate',
 'rush_epa_per_play',
 'rush_success_rate',
 'yards_per_rush',
 'early_down_pass_rate',
 'early_down_pass_success_rate',
 'early_down_epa_per_play',
 'early_down_pass_epa',
 'pressure_allowed_rate',
 'sack_rate',
 'qb_hit_rate_allowed',
 'negative_play_rate',
 'turnover_rate',
 'explosive_play_rate',
 'explosive_pass_rate',
 'explosive_rush_rate',
 'third_down_attempts',
 'third_down_conversions',
 'fourth_down_attempts',
 'fourth_down_conversions',
 'avg_xyac_epa',
 'avg_xyac_yards',
 'avg_xyac_success',
 'third_down_conversion_rate',
 'fourth_down_conversion_rate',
 'defensive_plays',
 'def_epa_allowed_per_play',
 'def_success_rate_allowed',
 'def_yards_allowed_per_play',
 'def_pass_epa_allowed',
 'def_pass_success_allowed',
 'def_yards_allowe

In [ ]:
# Identify rate-based features for review
rate_features = [
    col for col in selected_team_features
    if "rate" in col.lower()
    or "success" in col.lower()
    or "epa" in col.lower()
    or "conversion" in col.lower()
]

print("Number of rate / efficiency features:", len(rate_features))
rate_features

Number of rate / efficiency features: 43


['off_epa_per_play',
 'off_success_rate',
 'pass_rate',
 'pass_epa_per_play',
 'pass_success_rate',
 'rush_rate',
 'rush_epa_per_play',
 'rush_success_rate',
 'early_down_pass_rate',
 'early_down_pass_success_rate',
 'early_down_epa_per_play',
 'early_down_pass_epa',
 'pressure_allowed_rate',
 'sack_rate',
 'qb_hit_rate_allowed',
 'negative_play_rate',
 'turnover_rate',
 'explosive_play_rate',
 'explosive_pass_rate',
 'explosive_rush_rate',
 'third_down_conversions',
 'fourth_down_conversions',
 'avg_xyac_epa',
 'avg_xyac_success',
 'third_down_conversion_rate',
 'fourth_down_conversion_rate',
 'def_epa_allowed_per_play',
 'def_success_rate_allowed',
 'def_pass_epa_allowed',
 'def_pass_success_allowed',
 'def_rush_epa_allowed',
 'def_rush_success_allowed',
 'def_early_down_epa_allowed',
 'def_early_down_pass_epa_allowed',
 'def_early_down_pass_success_allowed',
 'pressure_created_rate',
 'sack_created_rate',
 'qb_hit_created_rate',
 'tackle_for_loss_rate',
 'explosive_allowed_rate',
 '

In [ ]:
# Create pregame rolling averages by team and season.
# The shift(1) prevents the current game from being included in its own prediction features.

for col in selected_team_features:
    team_game_stats[f"{col}_pregame"] = (
        team_game_stats
        .groupby(["team", "season"])[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )

pregame_feature_cols = [f"{col}_pregame" for col in selected_team_features]

team_game_stats[
    ["game_id", "season", "week", "game_date", "team"] + pregame_feature_cols[:10]
].head(20)

,game_id,season,week,game_date,team,offensive_plays_pregame,off_epa_per_play_pregame,off_success_rate_pregame,yards_per_play_pregame,pass_rate_pregame,pass_epa_per_play_pregame,pass_success_rate_pregame,yards_per_pass_pregame,avg_air_yards_pregame,avg_yac_pregame
0,2010_01_ARI_STL,2010,1,2010-09-12,ARI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32,2010_02_ARI_ATL,2010,2,2010-09-19,ARI,64.000,-0.123,0.359,5.906,0.672,0.066,0.372,6.186,10.220,5.955
82,2010_03_OAK_ARI,2010,3,2010-09-26,ARI,56.000,-0.218,0.336,5.734,0.701,-0.226,0.372,5.222,10.595,5.088
96,2010_04_ARI_SD,2010,4,2010-10-03,ARI,55.333,-0.197,0.341,5.224,0.640,-0.225,0.367,4.767,10.076,4.587
138,2010_05_NO_ARI,2010,5,2010-10-10,ARI,54.250,-0.271,0.349,4.526,0.661,-0.333,0.363,4.095,9.696,3.907
180,2010_07_ARI_SEA,2010,7,2010-10-24,ARI,54.800,-0.254,0.346,4.301,0.645,-0.272,0.387,4.204,9.295,3.770
228,2010_08_TB_ARI,2010,8,2010-10-31,ARI,54.833,-0.260,0.352,4.272,0.643,-0.301,0.370,4.046,9.144,3.933
234,2010_09_ARI_MIN,2010,9,2010-11-07,ARI,56.143,-0.257,0.366,4.546,0.643,-0.318,0.384,4.499,9.217,4.097
282,2010_10_SEA_ARI,2010,10,2010-11-14,ARI,55.875,-0.234,0.362,4.499,0.639,-0.279,0.377,4.588,9.356,3.999
288,2010_11_ARI_KC,2010,11,2010-11-21,ARI,56.889,-0.232,0.363,4.561,0.655,-0.274,0.381,4.706,9.341,4.111


## Missing Values in Pregame Rolling Features

Because the pregame features are calculated using only games played before the current game, Week 1 observations will naturally have missing values. This is expected because teams have not played any prior games in the same season. Later in the project, these missing values can be handled by dropping early-season games, imputing with league averages, or using previous-season information.

In [ ]:
# Check missing values in the pregame rolling features
pregame_missing = (
    team_game_stats[pregame_feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

pregame_missing.head(25)

,0
fourth_down_conversion_rate_pregame,851
off_epa_per_play_pregame,512
offensive_plays_pregame,512
yards_per_play_pregame,512
pass_rate_pregame,512
pass_epa_per_play_pregame,512
pass_success_rate_pregame,512
yards_per_pass_pregame,512
avg_air_yards_pregame,512
avg_yac_pregame,512


In [ ]:
# Check how many team-game rows occur by week
team_game_stats["week"].value_counts().sort_index().head(10)

,count
week,
1,510
2,512
3,512
4,488
5,460
6,458
7,452
8,452
9,434


## Creating Home and Away Pregame Feature Tables

The team-game advanced statistics dataset has one row per team per game. To create a final game-level dataset, I split these team-level pregame features into separate home-team and away-team feature tables. These will then be merged back onto the game-level schedule and betting dataset.

In [ ]:
# Keep only identifiers and pregame features
team_pregame = team_game_stats[
    ["game_id", "team"] + pregame_feature_cols
].copy()

# Create home-team feature table
home_features = team_pregame.copy()
home_features = home_features.rename(columns={"team": "home_team"})

home_features = home_features.rename(
    columns={col: f"home_{col.replace('_pregame', '')}" for col in pregame_feature_cols}
)

# Create away-team feature table
away_features = team_pregame.copy()
away_features = away_features.rename(columns={"team": "away_team"})

away_features = away_features.rename(
    columns={col: f"away_{col.replace('_pregame', '')}" for col in pregame_feature_cols}
)

print("Home features shape:", home_features.shape)
print("Away features shape:", away_features.shape)

Home features shape: (8350, 61)
Away features shape: (8350, 61)


## Creating the Game-Level EDA Dataset

The final dataset should be tidy, meaning each row represents one observation. For this project, one observation is one NFL game. I merged the home team's pregame features and the away team's pregame features onto the completed regular season games dataset.

In [ ]:
# Start with the completed regular season games dataset
eda_df = games.copy()

# Merge home-team pregame features
eda_df = eda_df.merge(
    home_features,
    on=["game_id", "home_team"],
    how="left"
)

# Merge away-team pregame features
eda_df = eda_df.merge(
    away_features,
    on=["game_id", "away_team"],
    how="left"
)

print("Original games shape:", games.shape)
print("EDA dataset shape after merging home and away features:", eda_df.shape)

eda_df.head()

Original games shape: (4175, 52)
EDA dataset shape after merging home and away features: (4175, 170)


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium,home_margin,market_home_margin,home_spread,spread_result,home_cover,push,home_offensive_plays,home_off_epa_per_play,home_off_success_rate,home_yards_per_play,home_pass_rate,home_pass_epa_per_play,home_pass_success_rate,home_yards_per_pass,home_avg_air_yards,home_avg_yac,home_avg_cpoe,home_avg_xpass,home_avg_pass_oe,home_rush_rate,home_rush_epa_per_play,home_rush_success_rate,home_yards_per_rush,home_early_down_pass_rate,home_early_down_pass_success_rate,home_early_down_epa_per_play,home_early_down_pass_epa,home_pressure_allowed_rate,home_sack_rate,home_qb_hit_rate_allowed,home_negative_play_rate,home_turnover_rate,home_explosive_play_rate,home_explosive_pass_rate,home_explosive_rush_rate,home_third_down_attempts,home_third_down_conversions,home_fourth_down_attempts,home_fourth_down_conversions,home_avg_xyac_epa,home_avg_xyac_yards,home_avg_xyac_success,home_third_down_conversion_rate,home_fourth_down_conversion_rate,home_defensive_plays,home_def_epa_allowed_per_play,home_def_success_rate_allowed,home_def_yards_allowed_per_play,home_def_pass_epa_allowed,home_def_pass_success_allowed,home_def_yards_allowed_per_pass,home_def_rush_epa_allowed,home_def_rush_success_allowed,home_def_yards_allowed_per_rush,home_def_early_down_epa_allowed,home_def_early_down_pass_epa_allowed,home_def_early_down_pass_success_allowed,home_pressure_created_rate,home_sack_created_rate,home_qb_hit_created_rate,home_tackle_for_loss_rate,home_explosive_allowed_rate,home_explosive_pass_allowed_rate,home_explosive_rush_allowed_rate,home_turnover_forced_rate,away_offensive_plays,away_off_epa_per_play,away_off_success_rate,away_yards_per_play,away_pass_rate,away_pass_epa_per_play,away_pass_success_rate,away_yards_per_pass,away_avg_air_yards,away_avg_yac,away_avg_cpoe,away_avg_xpass,away_avg_pass_oe,away_rush_rate,away_rush_epa_per_play,away_rush_success_rate,away_yards_per_rush,away_early_down_pass_rate,away_early_down_pass_success_rate,away_early_down_epa_per_play,away_early_down_pass_epa,away_pressure_allowed_rate,away_sack_rate,away_qb_hit_rate_allowed,away_negative_play_rate,away_turnover_rate,away_explosive_play_rate,away_explosive_pass_rate,away_explosive_rush_rate,away_third_down_attempts,away_third_down_conversions,away_fourth_down_attempts,away_fourth_down_conversions,away_avg_xyac_epa,away_avg_xyac_yards,away_avg_xyac_success,away_third_down_conversion_rate,away_fourth_down_conversion_rate,away_defensive_plays,away_def_epa_allowed_per_play,away_def_success_rate_allowed,away_def_yards_allowed_per_play,away_def_pass_epa_allowed,away_def_pass_success_allowed,away_def_yards_allowed_per_pass,away_def_rush_epa_allowed,away_def_rush_success_allowed,away_def_yards_allowed_per_rush,away_def_early_down_epa_allowed,away_def_early_down_pass_epa_allowed,away_def_early_down_pass_success_allowed,away_pressure_created_rate,away_sack_created_rate,away_qb_hit_created_rate,away_tackle_for_loss_rate,away_explosive_allowed_rate,away_explosive_pass_allowed_rate,away_explosive_rush_allowed_rate,away_turnover_forced_rate
0,2010_01_MIN_NO,2010,REG,1,2010-09-09,Thursday,20:30,MIN,9.000,NO,14.000,Home,5.000,23.000,0.000,2010090900,54863.000,NaN,201009090nor,1727.000,300909018.000,NaN,7,7,197.000,-220.000,4.500,-105.000,-103.000,48.500,-104.000,-106.000,0,dome,sportturf,NaN,NaN,00-0005106,00-0020531,Brett Favre,Drew Brees,Brad Childress,Sean Payton,Terry McAulay,NOR00,Louisiana Superdome,5.000,4.500,-4.500,0.500,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [ ]:
# Start with the completed regular season games dataset
eda_df = games.copy()

# Merge home-team pregame features
eda_df = eda_df.merge(
    home_features,
    on=["game_id", "home_team"],
    how="left"
)

# Merge away-team pregame features
eda_df = eda_df.merge(
    away_features,
    on=["game_id", "away_team"],
    how="left"
)

print("Original games shape:", games.shape)
print("EDA dataset shape after merging home and away features:", eda_df.shape)

eda_df.head()

Original games shape: (4175, 52)
EDA dataset shape after merging home and away features: (4175, 170)


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium,home_margin,market_home_margin,home_spread,spread_result,home_cover,push,home_offensive_plays,home_off_epa_per_play,home_off_success_rate,home_yards_per_play,home_pass_rate,home_pass_epa_per_play,home_pass_success_rate,home_yards_per_pass,home_avg_air_yards,home_avg_yac,home_avg_cpoe,home_avg_xpass,home_avg_pass_oe,home_rush_rate,home_rush_epa_per_play,home_rush_success_rate,home_yards_per_rush,home_early_down_pass_rate,home_early_down_pass_success_rate,home_early_down_epa_per_play,home_early_down_pass_epa,home_pressure_allowed_rate,home_sack_rate,home_qb_hit_rate_allowed,home_negative_play_rate,home_turnover_rate,home_explosive_play_rate,home_explosive_pass_rate,home_explosive_rush_rate,home_third_down_attempts,home_third_down_conversions,home_fourth_down_attempts,home_fourth_down_conversions,home_avg_xyac_epa,home_avg_xyac_yards,home_avg_xyac_success,home_third_down_conversion_rate,home_fourth_down_conversion_rate,home_defensive_plays,home_def_epa_allowed_per_play,home_def_success_rate_allowed,home_def_yards_allowed_per_play,home_def_pass_epa_allowed,home_def_pass_success_allowed,home_def_yards_allowed_per_pass,home_def_rush_epa_allowed,home_def_rush_success_allowed,home_def_yards_allowed_per_rush,home_def_early_down_epa_allowed,home_def_early_down_pass_epa_allowed,home_def_early_down_pass_success_allowed,home_pressure_created_rate,home_sack_created_rate,home_qb_hit_created_rate,home_tackle_for_loss_rate,home_explosive_allowed_rate,home_explosive_pass_allowed_rate,home_explosive_rush_allowed_rate,home_turnover_forced_rate,away_offensive_plays,away_off_epa_per_play,away_off_success_rate,away_yards_per_play,away_pass_rate,away_pass_epa_per_play,away_pass_success_rate,away_yards_per_pass,away_avg_air_yards,away_avg_yac,away_avg_cpoe,away_avg_xpass,away_avg_pass_oe,away_rush_rate,away_rush_epa_per_play,away_rush_success_rate,away_yards_per_rush,away_early_down_pass_rate,away_early_down_pass_success_rate,away_early_down_epa_per_play,away_early_down_pass_epa,away_pressure_allowed_rate,away_sack_rate,away_qb_hit_rate_allowed,away_negative_play_rate,away_turnover_rate,away_explosive_play_rate,away_explosive_pass_rate,away_explosive_rush_rate,away_third_down_attempts,away_third_down_conversions,away_fourth_down_attempts,away_fourth_down_conversions,away_avg_xyac_epa,away_avg_xyac_yards,away_avg_xyac_success,away_third_down_conversion_rate,away_fourth_down_conversion_rate,away_defensive_plays,away_def_epa_allowed_per_play,away_def_success_rate_allowed,away_def_yards_allowed_per_play,away_def_pass_epa_allowed,away_def_pass_success_allowed,away_def_yards_allowed_per_pass,away_def_rush_epa_allowed,away_def_rush_success_allowed,away_def_yards_allowed_per_rush,away_def_early_down_epa_allowed,away_def_early_down_pass_epa_allowed,away_def_early_down_pass_success_allowed,away_pressure_created_rate,away_sack_created_rate,away_qb_hit_created_rate,away_tackle_for_loss_rate,away_explosive_allowed_rate,away_explosive_pass_allowed_rate,away_explosive_rush_allowed_rate,away_turnover_forced_rate
0,2010_01_MIN_NO,2010,REG,1,2010-09-09,Thursday,20:30,MIN,9.000,NO,14.000,Home,5.000,23.000,0.000,2010090900,54863.000,NaN,201009090nor,1727.000,300909018.000,NaN,7,7,197.000,-220.000,4.500,-105.000,-103.000,48.500,-104.000,-106.000,0,dome,sportturf,NaN,NaN,00-0005106,00-0020531,Brett Favre,Drew Brees,Brad Childress,Sean Payton,Terry McAulay,NOR00,Louisiana Superdome,5.000,4.500,-4.500,0.500,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

## Creating Differential Features

For many football statistics, the difference between the home team and away team may be more useful than either team's value alone. For example, a positive offensive EPA difference means the home team's offensive EPA was higher than the away team's offensive EPA entering the game.

Differential features are calculated as:

`home team pregame value - away team pregame value`

In [ ]:
# Create differential features for every selected team feature
# Each differential feature is home value minus away value

for col in selected_team_features:
    home_col = f"home_{col}"
    away_col = f"away_{col}"
    diff_col = f"{col}_diff"

    if home_col in eda_df.columns and away_col in eda_df.columns:
        eda_df[diff_col] = eda_df[home_col] - eda_df[away_col]

# List all created differential features
diff_cols = [col for col in eda_df.columns if col.endswith("_diff")]

print("Number of differential features created:", len(diff_cols))

eda_df[
    [
        "game_id",
        "season",
        "week",
        "home_team",
        "away_team",
        "home_margin",
        "market_home_margin"
    ] + diff_cols[:10]
].head()

Number of differential features created: 59


,game_id,season,week,home_team,away_team,home_margin,market_home_margin,offensive_plays_diff,off_epa_per_play_diff,off_success_rate_diff,yards_per_play_diff,pass_rate_diff,pass_epa_per_play_diff,pass_success_rate_diff,yards_per_pass_diff,avg_air_yards_diff,avg_yac_diff
0,2010_01_MIN_NO,2010,1,NO,MIN,5.000,4.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010_01_MIA_BUF,2010,1,BUF,MIA,-5.000,-3.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010_01_DET_CHI,2010,1,CHI,DET,5.000,6.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010_01_IND_HOU,2010,1,HOU,IND,10.000,-1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010_01_DEN_JAX,2010,1,JAX,DEN,7.000,3.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Selecting Final EDA Columns

The original nflverse play-by-play dataset contains hundreds of columns. For this EDA, I kept the columns most relevant to the project: game identifiers, betting market variables, rest/weather/context variables, the target variable, home-team pregame features, away-team pregame features, and home-away differential features.

This keeps the dataset broad enough for later feature selection while still removing unrelated play-level columns.

In [ ]:
# Core game-level columns
core_cols = [
    "game_id",
    "season",
    "week",
    "gameday",
    "weekday",
    "away_team",
    "home_team",
    "away_score",
    "home_score",
    "home_margin",
    "home_spread",
    "market_home_margin",
    "spread_result",
    "home_cover",
    "push",
    "total_line",
    "home_rest",
    "away_rest",
    "roof",
    "surface",
    "temp",
    "wind",
    "div_game",
    "stadium"
]

# All home, away, and differential features
home_team_cols = [col for col in eda_df.columns if col.startswith("home_") and col not in core_cols]
away_team_cols = [col for col in eda_df.columns if col.startswith("away_") and col not in core_cols]
diff_cols = [col for col in eda_df.columns if col.endswith("_diff")]

# Final EDA columns
final_eda_cols = [
    col for col in core_cols + home_team_cols + away_team_cols + diff_cols
    if col in eda_df.columns
]

eda_final = eda_df[final_eda_cols].copy()

print("Final EDA dataset shape:", eda_final.shape)
eda_final.head()

Final EDA dataset shape: (4175, 211)


,game_id,season,week,gameday,weekday,away_team,home_team,away_score,home_score,home_margin,home_spread,market_home_margin,spread_result,home_cover,push,total_line,home_rest,away_rest,roof,surface,temp,wind,div_game,stadium,home_moneyline,home_spread_odds,home_qb_id,home_qb_name,home_coach,home_offensive_plays,home_off_epa_per_play,home_off_success_rate,home_yards_per_play,home_pass_rate,home_pass_epa_per_play,home_pass_success_rate,home_yards_per_pass,home_avg_air_yards,home_avg_yac,home_avg_cpoe,home_avg_xpass,home_avg_pass_oe,home_rush_rate,home_rush_epa_per_play,home_rush_success_rate,home_yards_per_rush,home_early_down_pass_rate,home_early_down_pass_success_rate,home_early_down_epa_per_play,home_early_down_pass_epa,home_pressure_allowed_rate,home_sack_rate,home_qb_hit_rate_allowed,home_negative_play_rate,home_turnover_rate,home_explosive_play_rate,home_explosive_pass_rate,home_explosive_rush_rate,home_third_down_attempts,home_third_down_conversions,home_fourth_down_attempts,home_fourth_down_conversions,home_avg_xyac_epa,home_avg_xyac_yards,home_avg_xyac_success,home_third_down_conversion_rate,home_fourth_down_conversion_rate,home_defensive_plays,home_def_epa_allowed_per_play,home_def_success_rate_allowed,home_def_yards_allowed_per_play,home_def_pass_epa_allowed,home_def_pass_success_allowed,home_def_yards_allowed_per_pass,home_def_rush_epa_allowed,home_def_rush_success_allowed,home_def_yards_allowed_per_rush,home_def_early_down_epa_allowed,home_def_early_down_pass_epa_allowed,home_def_early_down_pass_success_allowed,home_pressure_created_rate,home_sack_created_rate,home_qb_hit_created_rate,home_tackle_for_loss_rate,home_explosive_allowed_rate,home_explosive_pass_allowed_rate,home_explosive_rush_allowed_rate,home_turnover_forced_rate,away_moneyline,away_spread_odds,away_qb_id,away_qb_name,away_coach,away_offensive_plays,away_off_epa_per_play,away_off_success_rate,away_yards_per_play,away_pass_rate,away_pass_epa_per_play,away_pass_success_rate,away_yards_per_pass,away_avg_air_yards,away_avg_yac,away_avg_cpoe,away_avg_xpass,away_avg_pass_oe,away_rush_rate,away_rush_epa_per_play,away_rush_success_rate,away_yards_per_rush,away_early_down_pass_rate,away_early_down_pass_success_rate,away_early_down_epa_per_play,away_early_down_pass_epa,away_pressure_allowed_rate,away_sack_rate,away_qb_hit_rate_allowed,away_negative_play_rate,away_turnover_rate,away_explosive_play_rate,away_explosive_pass_rate,away_explosive_rush_rate,away_third_down_attempts,away_third_down_conversions,away_fourth_down_attempts,away_fourth_down_conversions,away_avg_xyac_epa,away_avg_xyac_yards,away_avg_xyac_success,away_third_down_conversion_rate,away_fourth_down_conversion_rate,away_defensive_plays,away_def_epa_allowed_per_play,away_def_success_rate_allowed,away_def_yards_allowed_per_play,away_def_pass_epa_allowed,away_def_pass_success_allowed,away_def_yards_allowed_per_pass,away_def_rush_epa_allowed,away_def_rush_success_allowed,away_def_yards_allowed_per_rush,away_def_early_down_epa_allowed,away_def_early_down_pass_epa_allowed,away_def_early_down_pass_success_allowed,away_pressure_created_rate,away_sack_created_rate,away_qb_hit_created_rate,away_tackle_for_loss_rate,away_explosive_allowed_rate,away_explosive_pass_allowed_rate,away_explosive_rush_allowed_rate,away_turnover_forced_rate,offensive_plays_diff,off_epa_per_play_diff,off_success_rate_diff,yards_per_play_diff,pass_rate_diff,pass_epa_per_play_diff,pass_success_rate_diff,yards_per_pass_diff,avg_air_yards_diff,avg_yac_diff,avg_cpoe_diff,avg_xpass_diff,avg_pass_oe_diff,rush_rate_diff,rush_epa_per_play_diff,rush_success_rate_diff,yards_per_rush_diff,early_down_pass_rate_diff,early_down_pass_success_rate_diff,early_down_epa_per_play_diff,early_down_pass_epa_diff,pressure_allowed_rate_diff,sack_rate_diff,qb_hit_rate_allowed_diff,negative_play_rate_diff,turnover_rate_diff,explosive_play_rate_diff,explosive_pass_rate_diff,explosive_rush_rate_diff,third_down_attempts_diff,third_down_conversions_diff,fourth_down_

In [ ]:
# Save the final tidy EDA dataset
eda_file = processed_dir / "nfl_spread_eda_dataset_2010_2025.csv"

eda_final.to_csv(eda_file, index=False)

print(f"Saved final EDA dataset to: {eda_file}")

Saved final EDA dataset to: /content/nfl_spread_capstone/data/processed/nfl_spread_eda_dataset_2010_2025.csv


## Summary Statistics

The following summary statistics focus on the target variable, sportsbook market variables, rest/weather variables, and selected advanced performance differential features.

In [ ]:
# Summary statistics for core numeric variables
core_numeric_cols = [
    "home_margin",
    "home_spread",
    "market_home_margin",
    "spread_result",
    "total_line",
    "home_rest",
    "away_rest",
    "temp",
    "wind"
]

core_numeric_cols = [col for col in core_numeric_cols if col in eda_final.columns]

eda_final[core_numeric_cols].describe()

,home_margin,home_spread,market_home_margin,spread_result,total_line,home_rest,away_rest,temp,wind
count,4175.000,4175.000,4175.000,4175.000,4175.000,4175.000,4175.000,2827.000,2827.000
mean,2.015,-1.954,1.954,0.061,44.962,7.377,7.489,58.514,8.318
std,14.498,5.974,5.974,13.060,4.352,1.989,2.067,16.830,5.162
min,-49.000,-27.000,-18.000,-52.000,28.500,4.000,4.000,1.000,0.000
25%,-7.000,-6.000,-3.000,-8.000,42.000,7.000,7.000,46.000,5.000
50%,3.000,-3.000,3.000,0.000,44.500,7.000,7.000,59.000,8.000
75%,10.000,3.000,6.000,8.000,47.500,7.000,7.000,71.000,11.000
max,58.000,18.000,27.000,47.500,63.500,16.000,17.000,97.000,71.000


In [ ]:
# Summary statistics for differential features
eda_final[diff_cols].describe().T.head(30)

,count,mean,std,min,25%,50%,75%,max
offensive_plays_diff,3590.000,-0.161,6.184,-34.000,-3.667,0.000,3.551,32.000
off_epa_per_play_diff,3590.000,0.002,0.159,-0.896,-0.102,0.004,0.102,0.847
off_success_rate_diff,3590.000,0.000,0.060,-0.382,-0.039,-0.000,0.038,0.273
yards_per_play_diff,3590.000,-0.007,0.934,-4.570,-0.612,-0.002,0.581,5.501
pass_rate_diff,3590.000,0.001,0.086,-0.347,-0.051,0.001,0.055,0.402
pass_epa_per_play_diff,3590.000,0.001,0.237,-1.242,-0.148,-0.001,0.143,1.531
pass_success_rate_diff,3590.000,-0.001,0.077,-0.346,-0.049,-0.001,0.048,0.420
yards_per_pass_diff,3590.000,-0.024,1.468,-6.433,-0.929,-0.003,0.841,11.677
avg_air_yards_diff,3590.000,0.005,1.660,-9.209,-1.017,0.019,1.034,7.865
avg_yac_diff,3590.000,-0.008,1.262,-6.154,-0.766,0.006,0.732,7.690


## Missing Value Analysis

Missing values are important because machine learning models generally require complete numeric input. Some missing values are expected in this project because Week 1 games do not have current-season pregame rolling averages. Weather variables may also contain missing values for dome games, games with unavailable weather data, or older records.

In [ ]:
# Missing values by column
missing_summary = (
    eda_final
    .isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(eda_final) * 100
)

missing_summary = missing_summary.sort_values("missing_percent", ascending=False)

missing_summary.head(30)

,column,missing_count,missing_percent
20,temp,1348,32.287
21,wind,1348,32.287
189,fourth_down_conversion_rate_diff,848,20.311
130,away_fourth_down_conversion_rate,601,14.395
204,sack_created_rate_diff,585,14.012
209,explosive_rush_allowed_rate_diff,585,14.012
206,tackle_for_loss_rate_diff,585,14.012
207,explosive_allowed_rate_diff,585,14.012
200,def_early_down_epa_allowed_diff,585,14.012
205,qb_hit_created_rate_diff,585,14.012


## Duplicate Row Analysis

Because the final EDA dataset should have one row per game, I checked for duplicate rows and duplicate game IDs. Duplicate game IDs would indicate a problem in the merge between the game-level dataset and the team-level advanced statistics.

In [ ]:
# Check for fully duplicated rows
duplicate_rows = eda_final.duplicated().sum()

# Check for duplicate game IDs
duplicate_game_ids = eda_final["game_id"].duplicated().sum()

print("Fully duplicated rows:", duplicate_rows)
print("Duplicate game IDs:", duplicate_game_ids)
print("Unique game IDs:", eda_final["game_id"].nunique())
print("Total rows:", eda_final.shape[0])

Fully duplicated rows: 0
Duplicate game IDs: 0
Unique game IDs: 4175
Total rows: 4175


## Target Variable: Home Margin

The target variable for the future machine learning model is `home_margin`, which is calculated as home score minus away score. Positive values mean the home team won, while negative values mean the away team won.

In [ ]:
fig = px.histogram(
    eda_final,
    x="home_margin",
    nbins=40,
    title="Distribution of Home Team Margin",
    labels={"home_margin": "Home Margin"}
)

fig.show()

In [ ]:
fig = px.box(
    eda_final,
    x="season",
    y="home_margin",
    title="Home Margin by Season",
    labels={"season": "Season", "home_margin": "Home Margin"}
)

fig.show()

### Interpretation

The home margin distribution shows how often home teams win or lose by different point margins. Because NFL games can have large blowouts but many games are decided by one score, the distribution is expected to be centered near zero with tails on both sides. The season-level boxplot helps identify whether the target variable is relatively stable across seasons.

## Sportsbook Market Expected Margin vs Actual Margin

The sportsbook spread can be converted into a market expected home margin. Comparing `market_home_margin` to `home_margin` shows how closely betting markets estimate actual game outcomes.

In [ ]:
fig = px.scatter(
    eda_final,
    x="market_home_margin",
    y="home_margin",
    hover_data=["season", "week", "home_team", "away_team"],
    title="Market Expected Home Margin vs Actual Home Margin",
    labels={
        "market_home_margin": "Market Expected Home Margin",
        "home_margin": "Actual Home Margin"
    }
)

fig.show()

In [ ]:
# Correlation between market expected margin and actual home margin
eda_final[["market_home_margin", "home_margin"]].corr()

,market_home_margin,home_margin
market_home_margin,1.000,0.435
home_margin,0.435,1.000


## Spread Result and Home Cover Rate

The variable `spread_result` measures how the home team performed against the sportsbook spread. It is calculated as actual home margin minus the market expected home margin.

If `spread_result` is greater than zero, the home team covered the spread. If it equals zero, the game was a push.

In [ ]:
# Home cover and push rates
home_cover_rate = eda_final.loc[eda_final["push"] == 0, "home_cover"].mean()
push_rate = eda_final["push"].mean()

print(f"Home cover rate excluding pushes: {home_cover_rate:.3f}")
print(f"Push rate: {push_rate:.3f}")

Home cover rate excluding pushes: 0.491
Push rate: 0.025


In [ ]:
fig = px.histogram(
    eda_final,
    x="spread_result",
    nbins=40,
    title="Distribution of Home Team Performance Against the Spread",
    labels={"spread_result": "Actual Home Margin - Market Expected Home Margin"}
)

fig.show()

### Interpretation

The spread result distribution shows how much the home team overperformed or underperformed compared with the sportsbook spread. Values above zero mean the home team covered, values below zero mean the away team covered, and values equal to zero represent pushes. A distribution centered near zero would suggest that sportsbook lines are generally well-calibrated, which is expected in an efficient betting market.

## Correlation Analysis

Correlation analysis helps identify which numeric features have the strongest linear relationship with the target variable, `home_margin`. This does not prove causation, but it is useful for early feature screening before machine learning.

In [ ]:
# Select numeric columns for correlation analysis
numeric_cols = eda_final.select_dtypes(include=["number"]).columns.tolist()

# Remove columns that are outcomes or direct derivatives of the target
exclude_from_corr = [
    "away_score",
    "home_score",
    "home_margin",
    "spread_result",
    "home_cover",
    "push",
    "home_spread",
    "market_home_margin",
    "away_moneyline",
    "home_moneyline"
]

feature_corr_cols = [
    col for col in numeric_cols
    if col not in exclude_from_corr
]

# Calculate correlations with home_margin
corr_with_target = (
    eda_final[feature_corr_cols + ["home_margin"]]
    .corr(numeric_only=True)["home_margin"]
    .drop("home_margin")
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

corr_with_target.head(30)

,home_margin
off_epa_per_play_diff,0.298
off_success_rate_diff,0.281
pass_epa_per_play_diff,0.280
pass_success_rate_diff,0.269
avg_xpass_diff,-0.268
early_down_epa_per_play_diff,0.259
yards_per_play_diff,0.249
yards_per_pass_diff,0.242
early_down_pass_epa_diff,0.238
avg_cpoe_diff,0.224


In [ ]:
top_corr = corr_with_target.head(20).reset_index()
top_corr.columns = ["feature", "correlation_with_home_margin"]

fig = px.bar(
    top_corr,
    x="correlation_with_home_margin",
    y="feature",
    orientation="h",
    title="Top 20 Feature Correlations with Home Margin",
    labels={
        "correlation_with_home_margin": "Correlation with Home Margin",
        "feature": "Feature"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### Interpretation

The correlation results show which variables have the strongest linear relationship with final home margin. The sportsbook market expected margin is expected to be one of the strongest predictors because betting lines already reflect team strength, injuries, location, and public market information. Advanced team differential features with stronger correlations may be useful predictors in future machine learning models.

## Advanced Team Feature Relationships

The following plots explore relationships between selected advanced team differential features and the target variable. Differential features compare the home team's pregame value to the away team's pregame value.

In [ ]:
# Key features selected from the correlation results
key_features = [
    "off_epa_per_play_diff",
    "off_success_rate_diff",
    "pass_epa_per_play_diff",
    "pass_success_rate_diff",
    "early_down_epa_per_play_diff",
    "yards_per_play_diff",
    "yards_per_pass_diff",
    "early_down_pass_epa_diff",
    "avg_cpoe_diff",
    "home_off_epa_per_play",
    "third_down_conversion_rate_diff",
    "home_pass_epa_per_play",
    "home_off_success_rate",
    "qb_hit_rate_allowed_diff",
    "away_pass_epa_per_play",
    "away_off_success_rate",
    "pressure_allowed_rate_diff",
    "away_off_epa_per_play",
    "home_avg_xpass",
    "avg_xpass_diff"
]

# Keep only features that actually exist in the final EDA dataset
key_features = [col for col in key_features if col in eda_final.columns]

print("Key features available for visualization:")
key_features

Key features available for visualization:


['off_epa_per_play_diff',
 'off_success_rate_diff',
 'pass_epa_per_play_diff',
 'pass_success_rate_diff',
 'early_down_epa_per_play_diff',
 'yards_per_play_diff',
 'yards_per_pass_diff',
 'early_down_pass_epa_diff',
 'avg_cpoe_diff',
 'home_off_epa_per_play',
 'third_down_conversion_rate_diff',
 'home_pass_epa_per_play',
 'home_off_success_rate',
 'qb_hit_rate_allowed_diff',
 'away_pass_epa_per_play',
 'away_off_success_rate',
 'pressure_allowed_rate_diff',
 'away_off_epa_per_play',
 'home_avg_xpass',
 'avg_xpass_diff']

In [ ]:
# Create scatterplots comparing each selected feature to home margin
for feature in key_features:
    fig = px.scatter(
        eda_final,
        x=feature,
        y="home_margin",
        hover_data=["season", "week", "home_team", "away_team"],
        title=f"{feature} vs Home Margin",
        labels={
            feature: feature,
            "home_margin": "Home Margin"
        },
        trendline="ols"
    )

    fig.show()

### Interpretation

These charts compare the selected advanced team performance features against the target variable, `home_margin`. The features include EPA, success rate, passing efficiency, early-down efficiency, pressure allowed, quarterback hit rate, and expected pass rate.

The relationship between any single feature and home margin is expected to be noisy because NFL outcomes are influenced by many factors. However, features such as EPA differential, success rate differential, passing EPA differential, and early-down passing EPA differential are useful because they summarize team efficiency before the game. Even if individual scatterplots show moderate or weak relationships, these variables may still improve predictive performance when combined in a machine learning model.

## Categorical Variables

The dataset includes categorical context variables such as roof type, playing surface, division game indicator, home team, and away team. These variables may be useful in future modeling because game environment and team identity can influence scoring margin and spread outcomes.

In [ ]:
# Review categorical columns available in the EDA dataset
categorical_cols = eda_final.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

categorical_cols

['game_id',
 'gameday',
 'weekday',
 'away_team',
 'home_team',
 'roof',
 'surface',
 'stadium',
 'home_qb_id',
 'home_qb_name',
 'home_coach',
 'away_qb_id',
 'away_qb_name',
 'away_coach']

In [ ]:
# Count games by roof type
roof_counts = eda_final["roof"].value_counts(dropna=False).reset_index()
roof_counts.columns = ["roof", "count"]

fig = px.bar(
    roof_counts,
    x="roof",
    y="count",
    title="Games by Roof Type",
    labels={"roof": "Roof Type", "count": "Number of Games"}
)

fig.show()

### Interpretation

The roof type chart shows the distribution of games played outdoors, in a dome, or in retractable-roof stadiums. This variable may be relevant because weather conditions are more likely to affect outdoor games than dome games.

In [ ]:
# Count games by playing surface
surface_counts = eda_final["surface"].value_counts(dropna=False).reset_index()
surface_counts.columns = ["surface", "count"]

fig = px.bar(
    surface_counts,
    x="surface",
    y="count",
    title="Games by Playing Surface",
    labels={"surface": "Playing Surface", "count": "Number of Games"}
)

fig.show()

### Surface Cleaning

The original surface variable contained several labels that represented similar or duplicate field types. For example, `grass` appeared in multiple forms due to formatting differences, and `a_turf` was grouped with `astroturf`. Modern turf labels such as `fieldturf`, `matrixturf`, and `sportturf` were grouped together as `fieldturf`.

Cleaning this variable reduces category duplication and makes the surface-based EDA easier to interpret.

In [ ]:
# Create a cleaned surface variable for EDA
eda_final["surface_clean"] = (
    eda_final["surface"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Combine similar surface labels
surface_map = {
    "grass": "grass",
    "a_turf": "astroturf",
    "astroturf": "astroturf",
    "fieldturf": "fieldturf",
    "matrixturf": "fieldturf",
    "sportturf": "fieldturf",
    "astroplay": "astroturf"
}

eda_final["surface_clean"] = eda_final["surface_clean"].replace(surface_map)

# Check cleaned categories
eda_final["surface_clean"].value_counts(dropna=False)

,count
surface_clean,
grass,2341
fieldturf,1526
astroturf,264
nan,44


In [ ]:
# Count games by playing surface
surface_counts = eda_final["surface_clean"].value_counts(dropna=False).reset_index()
surface_counts.columns = ["surface_clean", "count"]

fig = px.bar(
    surface_counts,
    x="surface_clean",
    y="count",
    title="Games by Playing Surface",
    labels={"surface": "Playing Surface", "count": "Number of Games"}
)

fig.show()

### Interpretation

The surface chart shows how many games were played on different field types. Playing surface may matter because teams may perform differently on grass versus turf, especially for speed, passing, and injury-related factors.

In [ ]:
# Average home margin by roof type
roof_margin = (
    eda_final
    .groupby("roof", dropna=False)["home_margin"]
    .mean()
    .reset_index()
)

fig = px.bar(
    roof_margin,
    x="roof",
    y="home_margin",
    title="Average Home Margin by Roof Type",
    labels={"roof": "Roof Type", "home_margin": "Average Home Margin"}
)

fig.show()

In [ ]:
# Average home margin by playing surface
surface_margin = (
    eda_final
    .groupby("surface_clean", dropna=False)["home_margin"]
    .mean()
    .reset_index()
)

fig = px.bar(
    surface_margin,
    x="surface_clean",
    y="home_margin",
    title="Average Home Margin by Playing Surface",
    labels={"surface": "Playing Surface", "home_margin": "Average Home Margin"}
)

fig.show()

### Interpretation

Average home margin by roof and surface gives an initial view of whether game environment may relate to scoring margin. These comparisons are exploratory and should not be interpreted as causal. If these variables appear useful, they can be included as categorical predictors in later machine learning models. Turf fields seem to favor the home team greater than grass fields. This is similar for domed fields compared to non domes.

## Division Games

Division games may be different from non-division games because teams are more familiar with each other and play each other more frequently. This section compares home margin and spread result for division and non-division games.

In [ ]:
# Count division games
div_counts = eda_final["div_game"].value_counts(dropna=False).reset_index()
div_counts.columns = ["division_game", "count"]

fig = px.bar(
    div_counts,
    x="division_game",
    y="count",
    title="Division Games vs Non-Division Games",
    labels={"division_game": "Division Game", "count": "Number of Games"}
)

fig.show()

In [ ]:
# Average home margin by division game indicator
div_margin = (
    eda_final
    .groupby("div_game", dropna=False)[["home_margin", "spread_result"]]
    .mean()
    .reset_index()
)

div_margin

,div_game,home_margin,spread_result
0,0,2.175,0.193
1,1,1.742,-0.166


In [ ]:
fig = px.bar(
    div_margin,
    x="div_game",
    y="home_margin",
    title="Average Home Margin: Division vs Non-Division Games",
    labels={"div_game": "Division Game", "home_margin": "Average Home Margin"}
)

fig.show()

### Interpretation

Division games are included as a possible context variable because teams may perform differently against familiar opponents. The average margin comparison gives an initial look at whether division games have different outcomes than non-division games. It looks as though division games are more tightly contested than non-division games.

## Final Tidy EDA Dataset

The final EDA dataset is tidy because each row represents one NFL game and each column represents one property of that game. The dataset includes the target variable, betting market variables, game context variables, home-team pregame features, away-team pregame features, and home-away differential features.

This dataset will be used in the next stage of the project for feature selection and machine learning.

In [ ]:
# Save final tidy EDA dataset
eda_file = processed_dir / "nfl_spread_eda_dataset_2010_2025.csv"

eda_final.to_csv(eda_file, index=False)

print(f"Final EDA dataset saved to: {eda_file}")
print("Final EDA dataset shape:", eda_final.shape)

Final EDA dataset saved to: /content/nfl_spread_capstone/data/processed/nfl_spread_eda_dataset_2010_2025.csv
Final EDA dataset shape: (4175, 212)


In [ ]:
# Final summary for project report
print("Final EDA Summary")
print("-----------------")
print(f"Rows: {eda_final.shape[0]}")
print(f"Columns: {eda_final.shape[1]}")
print(f"Unique games: {eda_final['game_id'].nunique()}")
print(f"Seasons: {eda_final['season'].min()} to {eda_final['season'].max()}")
print("Target variable: home_margin")
print(f"Number of differential features: {len(diff_cols)}")
print(f"Correlation between market expected margin and actual margin: {eda_final[['market_home_margin', 'home_margin']].corr().iloc[0,1]:.3f}")
print(f"Home cover rate excluding pushes: {eda_final.loc[eda_final['push'] == 0, 'home_cover'].mean():.3f}")
print(f"Push rate: {eda_final['push'].mean():.3f}")

Final EDA Summary
-----------------
Rows: 4175
Columns: 212
Unique games: 4175
Seasons: 2010 to 2025
Target variable: home_margin
Number of differential features: 59
Correlation between market expected margin and actual margin: 0.435
Home cover rate excluding pushes: 0.491
Push rate: 0.025


## EDA Conclusion

The EDA confirms that the final dataset is structured at the game level, with one row per NFL game. The target variable is `home_margin`, and the sportsbook market expectation is represented by `market_home_margin`.

The sportsbook expected margin has a moderate positive relationship with actual home margin, which suggests that the market line is informative but not perfect. Advanced team features such as EPA, success rate, passing efficiency, early-down passing performance, pressure rate, and expected pass rate may provide additional predictive value when combined in a machine learning model.

The data requires additional preparation before modeling. Week 1 games contain missing pregame rolling statistics because no previous current-season games exist for those teams. Weather variables also contain missing values. These issues will need to be handled through filtering, imputation, or additional feature engineering in the modeling stage.